In [ ]:
!pip install -q rdkit xgboost signaturizer tf-keras

In [ ]:
"""
Tox21: chemical-only vs. multimodal (chemical + BioFeat) representations.

ADULT DATA ONLY. This tests whether biological features add predictive signal
on Tox21. It is NOT the adult->pediatric transfer experiment.

Three runs, identical except for the feature matrix:
  1. chem            Morgan + RDKit descriptors
  2. chem + bio      + Chemical Checker signatures
  3. chem + shuffled + signatures with rows permuted  (capacity control)

If (2) beats (1) but (3) also beats (1), the gain was extra columns, not biology.

Colab: paste each numbered section into its own cell.
"""

# ============================================================
# CELL 1 — setup  (restart the runtime after this, then re-run it)
# ============================================================
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # must precede any tensorflow import
# !pip install -q rdkit xgboost signaturizer tf-keras

import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

RDLogger.DisableLog("rdApp.*")

SEED = 0
N_FOLDS = 5
MIN_TEST_POS = 10          # skip a task/fold with too few positives to score
SPACES = ["B1", "B2", "B4", "B5", "C3", "C4", "C5", "D1", "D2"]
# A* = structural (already covered by Morgan/RDKit), E* = clinical outcomes
# (leakage risk against adverse-event labels). Both excluded on purpose.


# ============================================================
# CELL 2 — load and standardize
# ============================================================
URL = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz"
df = pd.read_csv(URL)
TASKS = [c for c in df.columns if c not in ("smiles", "mol_id")]

_lfc = rdMolStandardize.LargestFragmentChooser()
_unc = rdMolStandardize.Uncharger()


def standardize(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    try:
        mol = rdMolStandardize.Cleanup(mol)
        mol = _lfc.choose(mol)      # strip counterions / salts
        mol = _unc.uncharge(mol)    # neutralize
        return Chem.MolToSmiles(mol)
    except Exception:
        return None


df["std_smiles"] = df["smiles"].map(standardize)
print("unparseable:", df["std_smiles"].isna().sum())
df = df.dropna(subset=["std_smiles"])

# collapse duplicate structures; a 0.5 mean means the same structure was
# labeled both ways for that task -> treat as missing
dedup = df.groupby("std_smiles", as_index=False)[TASKS].mean()
for t in TASKS:
    v = dedup[t].values
    dedup[t] = np.where(np.isnan(v), np.nan,
                        np.where(v > 0.5, 1.0, np.where(v < 0.5, 0.0, np.nan)))
dedup = dedup[dedup[TASKS].notna().any(axis=1)].reset_index(drop=True)

# drop anything that fails the standardization round-trip, keeping dedup aligned
_mols = [Chem.MolFromSmiles(s) for s in dedup["std_smiles"]]
keep = [i for i, m in enumerate(_mols) if m is not None]
dedup = dedup.iloc[keep].reset_index(drop=True)
mols = [_mols[i] for i in keep]

SMILES = dedup["std_smiles"].tolist()
Y = dedup[TASKS].values.astype(float)
MASK = ~np.isnan(Y)                      # labels are masked, never imputed
print(f"compounds: {len(mols)}   labels present: {MASK.sum()}")


# ============================================================
# CELL 3 — chemical features
# ============================================================
_mfp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
X_fp = np.array([_mfp.GetFingerprintAsNumPy(m) for m in mols], dtype=np.float32)

desc = pd.DataFrame([Descriptors.CalcMolDescriptors(m) for m in mols])
# Ipc/AvgIpc grow exponentially with molecule size and overflow float32
for c in ("Ipc", "AvgIpc"):
    if c in desc.columns:
        desc[c] = np.log1p(desc[c].clip(lower=0))
desc = desc.replace([np.inf, -np.inf], np.nan)
desc = desc.loc[:, desc.isna().mean() < 0.05]     # drop mostly-missing columns
desc = desc.fillna(desc.median())
desc = desc.loc[:, desc.std() > 0]                # drop constant columns
desc = desc.clip(-1e30, 1e30)

X_desc = desc.values.astype(np.float32)
X_chem = np.hstack([X_fp, X_desc])
print(f"morgan {X_fp.shape}  desc {X_desc.shape}  chem {X_chem.shape}")
assert np.isfinite(X_chem).all()


# ============================================================
# CELL 4 — scaffold folds + evaluation
# ============================================================
scaf_ids = []
for m in mols:
    try:
        scaf_ids.append(MurckoScaffold.MurckoScaffoldSmiles(mol=m, includeChirality=False))
    except Exception:
        scaf_ids.append("")

FOLDS = list(GroupKFold(n_splits=N_FOLDS).split(np.arange(len(mols)), groups=scaf_ids))
print("fold sizes:", [len(te) for _, te in FOLDS])


def make_xgb(y_train):
    spw = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
    return XGBClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.6,
        scale_pos_weight=spw, tree_method="hist",
        eval_metric="logloss", n_jobs=-1, random_state=SEED,
    )


def run_cv(X, name):
    """Per-fold mean ROC-AUC and PR-AUC, reported with spread across folds."""
    roc_by_fold, pr_by_fold = [], []
    for tr_all, te_all in FOLDS:
        roc, pr = [], []
        for j in range(len(TASKS)):
            tr = tr_all[MASK[tr_all, j]]
            te = te_all[MASK[te_all, j]]
            ytr, yte = Y[tr, j], Y[te, j]
            if len(np.unique(ytr)) < 2 or len(np.unique(yte)) < 2 or yte.sum() < MIN_TEST_POS:
                continue
            clf = make_xgb(ytr).fit(X[tr], ytr)
            p = clf.predict_proba(X[te])[:, 1]
            roc.append(roc_auc_score(yte, p))
            pr.append(average_precision_score(yte, p))
        roc_by_fold.append(np.mean(roc))
        pr_by_fold.append(np.mean(pr))

    roc_by_fold, pr_by_fold = np.array(roc_by_fold), np.array(pr_by_fold)
    print(f"\n=== {name} ===")
    print(f"  ROC-AUC {roc_by_fold.mean():.4f} +/- {roc_by_fold.std():.4f}   "
          + "folds: " + ", ".join(f"{x:.3f}" for x in roc_by_fold))
    print(f"  PR-AUC  {pr_by_fold.mean():.4f} +/- {pr_by_fold.std():.4f}")
    return {"name": name, "roc": roc_by_fold, "pr": pr_by_fold}


unparseable: 8
compounds: 7587   labels present: 75521
morgan (7587, 2048)  desc (7587, 214)  chem (7587, 2262)
fold sizes: [1698, 1473, 1472, 1472, 1472]


In [ ]:
# ============================================================
# CELL 5A — persistent Google Drive checkpoint setup
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

import hashlib
import json
import os
import time
import numpy as np
import pandas as pd

# Protect against accidentally loading features generated for a different
# compound list or a different compound order.
DATASET_HASH = hashlib.sha256(
    "\n".join(SMILES).encode("utf-8")
).hexdigest()

BASE_DIR = "/content/drive/MyDrive/Tox21_multimodal_checkpoints"
RUN_DIR = os.path.join(BASE_DIR, DATASET_HASH[:12])
os.makedirs(RUN_DIR, exist_ok=True)

print("Dataset hash:", DATASET_HASH)
print("Checkpoint folder:", RUN_DIR)


def atomic_save_npy(path, array):
    """Save an array without leaving a corrupted final file if interrupted."""
    temporary_path = path + ".temporary.npy"
    np.save(temporary_path, array)
    os.replace(temporary_path, path)


def atomic_save_csv(path, dataframe):
    """Save a CSV using a temporary file and atomic rename."""
    temporary_path = path + ".temporary.csv"
    dataframe.to_csv(temporary_path, index=False)
    os.replace(temporary_path, path)


metadata = {
    "dataset_hash": DATASET_HASH,
    "n_compounds": len(SMILES),
    "n_tasks": len(TASKS),
    "tasks": list(TASKS),
    "n_folds": N_FOLDS,
    "spaces": list(SPACES),
    "seed": SEED,
}

metadata_path = os.path.join(RUN_DIR, "run_metadata.json")

if os.path.exists(metadata_path):
    with open(metadata_path, "r") as file:
        old_metadata = json.load(file)

    if old_metadata["dataset_hash"] != DATASET_HASH:
        raise RuntimeError("Checkpoint dataset does not match this dataset.")
else:
    temporary_metadata_path = metadata_path + ".temporary"

    with open(temporary_metadata_path, "w") as file:
        json.dump(metadata, file, indent=2)

    os.replace(temporary_metadata_path, metadata_path)

print("Persistent checkpoint system ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset hash: 27f4dc5e18e00f6e10d83fce74026d7cdef618851383db45bfcf67042f03aaaa
Checkpoint folder: /content/drive/MyDrive/Tox21_multimodal_checkpoints/27f4dc5e18e0
Persistent checkpoint system ready.


In [ ]:
# ============================================================
# CELL 5B — restart-safe Chemical Checker signatures
# ============================================================
from signaturizer import Signaturizer

print(
    "Smoke test:",
    Signaturizer("B4").predict(SMILES[:5]).signature.shape
)

bio_blocks = []

for space in SPACES:
    block_path = os.path.join(
        RUN_DIR,
        f"chemical_checker_{space}.npy"
    )

    if os.path.exists(block_path):
        block = np.load(block_path)

        if block.shape != (len(SMILES), 128):
            raise RuntimeError(
                f"Saved {space} block has incorrect shape: {block.shape}"
            )

        if not np.isfinite(block).all():
            raise RuntimeError(
                f"Saved {space} block contains NaN or infinity."
            )

        print(f"{space}: loaded from Drive")

    else:
        print(f"{space}: generating...")
        start = time.time()

        block = signaturize_with_retry(
            space,
            structures,
        )

        if block.shape != (len(SMILES), 128):
            raise RuntimeError(
                f"{space} generated incorrect shape: {block.shape}"
            )

        if not np.isfinite(block).all():
            raise RuntimeError(
                f"{space} contains NaN or infinity."
            )

        atomic_save_npy(block_path, block)

        elapsed = (time.time() - start) / 60
        print(f"{space}: generated and saved in {elapsed:.1f} minutes")

    bio_blocks.append(block)

X_bio = np.hstack(bio_blocks).astype(np.float32)

assert X_bio.shape == (len(SMILES), len(SPACES) * 128)
assert X_bio.shape[0] == X_chem.shape[0]
assert np.isfinite(X_bio).all()

# Also save the combined matrix for convenient future loading.
combined_bio_path = os.path.join(
    RUN_DIR,
    "X_bio_combined.npy"
)

if not os.path.exists(combined_bio_path):
    atomic_save_npy(combined_bio_path, X_bio)

print("Biological matrix:", X_bio.shape)
print("All biological features are saved persistently.")

RemoteDisconnected: Remote end closed connection without response

In [ ]:
# ============================================================
# CELL 6 — task-level resumable evaluation
# ============================================================
from sklearn.metrics import roc_auc_score, average_precision_score

assert X_chem.shape[0] == X_bio.shape[0] == Y.shape[0]
assert np.isfinite(X_chem).all()
assert np.isfinite(X_bio).all()

N_COMPOUNDS = len(Y)


def fold_task_indices(fold_id, task_idx):
    """Return labeled training and testing indices for one fold/task."""
    tr_all, te_all = FOLDS[fold_id]

    tr = tr_all[MASK[tr_all, task_idx]]
    te = te_all[MASK[te_all, task_idx]]

    return tr, te


def task_is_eligible(tr, te, task_idx):
    """Apply the same eligibility rule to every representation."""
    ytr = Y[tr, task_idx]
    yte = Y[te, task_idx]

    if len(np.unique(ytr)) < 2:
        return False

    if len(np.unique(yte)) < 2:
        return False

    n_positive = int(yte.sum())
    n_negative = int(len(yte) - yte.sum())

    return (
        n_positive >= MIN_TEST_POS
        and n_negative >= MIN_TEST_POS
    )


def summarize_oof(oof, representation_name):
    """Reconstruct fold-level and pooled metrics from saved predictions."""
    fold_rows = []

    for fold_id in range(len(FOLDS)):
        fold_roc = []
        fold_pr = []

        for task_idx, task in enumerate(TASKS):
            tr, te = fold_task_indices(fold_id, task_idx)

            if not task_is_eligible(tr, te, task_idx):
                continue

            available = np.isfinite(oof[te, task_idx])

            if not available.all():
                continue

            yte = Y[te, task_idx]
            pred = oof[te, task_idx]

            fold_roc.append(roc_auc_score(yte, pred))
            fold_pr.append(average_precision_score(yte, pred))

        fold_rows.append({
            "representation": representation_name,
            "fold": fold_id + 1,
            "tasks_scored": len(fold_roc),
            "roc_auc": (
                np.mean(fold_roc) if fold_roc else np.nan
            ),
            "pr_auc": (
                np.mean(fold_pr) if fold_pr else np.nan
            ),
        })

    task_rows = []

    for task_idx, task in enumerate(TASKS):
        valid = (
            MASK[:, task_idx]
            & np.isfinite(oof[:, task_idx])
        )

        y = Y[valid, task_idx]
        pred = oof[valid, task_idx]

        if len(y) == 0 or len(np.unique(y)) < 2:
            continue

        task_rows.append({
            "representation": representation_name,
            "task": task,
            "n": len(y),
            "n_positive": int(y.sum()),
            "positive_rate": float(y.mean()),
            "roc_auc": roc_auc_score(y, pred),
            "pr_auc": average_precision_score(y, pred),
        })

    fold_df = pd.DataFrame(fold_rows)
    task_df = pd.DataFrame(task_rows)

    return fold_df, task_df


def evaluate_representation_resumable(X, name, checkpoint_key):
    """
    Resume one representation from its saved OOF matrix.

    Predictions are saved to Drive after every completed task/fold fit.
    """
    oof_path = os.path.join(
        RUN_DIR,
        f"oof_{checkpoint_key}.npy"
    )

    fold_table_path = os.path.join(
        RUN_DIR,
        f"fold_results_{checkpoint_key}.csv"
    )

    task_table_path = os.path.join(
        RUN_DIR,
        f"task_results_{checkpoint_key}.csv"
    )

    if os.path.exists(oof_path):
        oof = np.load(oof_path)

        if oof.shape != Y.shape:
            raise RuntimeError(
                f"{checkpoint_key} checkpoint has wrong shape: "
                f"{oof.shape}"
            )

        print(f"\n{name}: loaded existing checkpoint")

    else:
        oof = np.full(
            Y.shape,
            np.nan,
            dtype=np.float32
        )
        atomic_save_npy(oof_path, oof)
        print(f"\n{name}: created new checkpoint")

    expected_fits = 0
    completed_fits = 0

    for fold_id in range(len(FOLDS)):
        for task_idx in range(len(TASKS)):
            tr, te = fold_task_indices(fold_id, task_idx)

            if not task_is_eligible(tr, te, task_idx):
                continue

            expected_fits += 1

            if np.isfinite(oof[te, task_idx]).all():
                completed_fits += 1

    print(
        f"{name}: {completed_fits}/{expected_fits} "
        "task-fold fits already saved"
    )

    start = time.time()

    for fold_id in range(len(FOLDS)):
        for task_idx, task in enumerate(TASKS):
            tr, te = fold_task_indices(fold_id, task_idx)

            if not task_is_eligible(tr, te, task_idx):
                continue

            # Skip a task/fold whose predictions are already in Drive.
            if np.isfinite(oof[te, task_idx]).all():
                print(
                    f"SKIP | {name} | fold {fold_id + 1}/5 | "
                    f"{task} already saved"
                )
                continue

            ytr = Y[tr, task_idx]

            print(
                f"FIT  | {name} | fold {fold_id + 1}/5 | "
                f"{task}"
            )

            clf = make_xgb(ytr)
            clf.fit(X[tr], ytr)

            pred = clf.predict_proba(X[te])[:, 1]
            oof[te, task_idx] = pred.astype(np.float32)

            # Persistent checkpoint after every fitted model.
            atomic_save_npy(oof_path, oof)

            completed_fits += 1

            print(
                f"SAVED | {name} | fold {fold_id + 1}/5 | "
                f"{task} | {completed_fits}/{expected_fits}"
            )

        # Save readable intermediate tables after every fold.
        fold_df, task_df = summarize_oof(oof, name)
        atomic_save_csv(fold_table_path, fold_df)
        atomic_save_csv(task_table_path, task_df)

        current_fold = fold_df.iloc[fold_id]

        print(
            f"\n{name} | fold {fold_id + 1}/5 checkpointed | "
            f"tasks={int(current_fold['tasks_scored'])} | "
            f"AUROC={current_fold['roc_auc']:.4f} | "
            f"AUPRC={current_fold['pr_auc']:.4f}\n"
        )

    fold_df, task_df = summarize_oof(oof, name)

    result = {
        "name": name,
        "checkpoint_key": checkpoint_key,
        "oof": oof,
        "fold_results": fold_df,
        "task_results": task_df,
        "macro_roc": task_df["roc_auc"].mean(),
        "macro_pr": task_df["pr_auc"].mean(),
        "fold_roc_mean": fold_df["roc_auc"].mean(),
        "fold_roc_std": fold_df["roc_auc"].std(ddof=1),
        "fold_pr_mean": fold_df["pr_auc"].mean(),
        "fold_pr_std": fold_df["pr_auc"].std(ddof=1),
    }

    elapsed = (time.time() - start) / 60

    print(f"\n=== {name} complete ===")
    print(f"Pooled task-macro AUROC: {result['macro_roc']:.4f}")
    print(f"Pooled task-macro AUPRC: {result['macro_pr']:.4f}")
    print(
        f"Fold AUROC: {result['fold_roc_mean']:.4f} "
        f"+/- {result['fold_roc_std']:.4f}"
    )
    print(f"Current-session runtime: {elapsed:.1f} minutes\n")

    return result

In [ ]:
# ============================================================
# CELL 7 — restart-safe representation experiments
# ============================================================

PERMUTATION_SEEDS = [101, 202, 303, 404, 505]

experiment_specs = [
    ("chem", "Chemical only", None),
    (
        "chem_true_bio",
        "Chemical + true biological signatures",
        None,
    ),
    (
        "bio_only",
        "Biological signatures only",
        None,
    ),
]

for seed in PERMUTATION_SEEDS:
    experiment_specs.append(
        (
            f"chem_permuted_bio_seed_{seed}",
            f"Chemical + permuted bio (seed {seed})",
            seed,
        )
    )

results = {}

for checkpoint_key, name, permutation_seed in experiment_specs:

    if checkpoint_key == "chem":
        X_current = X_chem

    elif checkpoint_key == "chem_true_bio":
        X_current = np.hstack(
            [X_chem, X_bio]
        ).astype(np.float32)

    elif checkpoint_key == "bio_only":
        X_current = X_bio

    else:
        rng = np.random.default_rng(permutation_seed)
        permutation = rng.permutation(N_COMPOUNDS)

        # Permute whole signature rows.
        X_permuted_bio = X_bio[permutation]

        X_current = np.hstack(
            [X_chem, X_permuted_bio]
        ).astype(np.float32)

    results[checkpoint_key] = evaluate_representation_resumable(
        X_current,
        name,
        checkpoint_key,
    )

    # Update the master summary after every representation.
    summary_rows = []

    for key, result in results.items():
        summary_rows.append({
            "key": key,
            "representation": result["name"],
            "pooled_macro_roc_auc": result["macro_roc"],
            "pooled_macro_pr_auc": result["macro_pr"],
            "fold_roc_mean": result["fold_roc_mean"],
            "fold_roc_std": result["fold_roc_std"],
            "fold_pr_mean": result["fold_pr_mean"],
            "fold_pr_std": result["fold_pr_std"],
        })

    running_summary = pd.DataFrame(summary_rows)

    atomic_save_csv(
        os.path.join(RUN_DIR, "representation_summary.csv"),
        running_summary,
    )

    print(
        f"Master summary updated after: {name}"
    )

    # Release concatenated matrices before constructing the next one.
    if checkpoint_key not in ("chem", "bio_only"):
        del X_current

    if permutation_seed is not None:
        del X_permuted_bio

print("\nAll representation experiments complete.")
print(running_summary.to_string(index=False))
print("\nSaved in:", RUN_DIR)


Chemical only: loaded existing checkpoint
Chemical only: 1/60 task-fold fits already saved
SKIP | Chemical only | fold 1/5 | NR-AR already saved
FIT  | Chemical only | fold 1/5 | NR-AR-LBD
SAVED | Chemical only | fold 1/5 | NR-AR-LBD | 2/60
FIT  | Chemical only | fold 1/5 | NR-AhR
SAVED | Chemical only | fold 1/5 | NR-AhR | 3/60
FIT  | Chemical only | fold 1/5 | NR-Aromatase
SAVED | Chemical only | fold 1/5 | NR-Aromatase | 4/60
FIT  | Chemical only | fold 1/5 | NR-ER
SAVED | Chemical only | fold 1/5 | NR-ER | 5/60
FIT  | Chemical only | fold 1/5 | NR-ER-LBD
SAVED | Chemical only | fold 1/5 | NR-ER-LBD | 6/60
FIT  | Chemical only | fold 1/5 | NR-PPAR-gamma
SAVED | Chemical only | fold 1/5 | NR-PPAR-gamma | 7/60
FIT  | Chemical only | fold 1/5 | SR-ARE
SAVED | Chemical only | fold 1/5 | SR-ARE | 8/60
FIT  | Chemical only | fold 1/5 | SR-ATAD5
SAVED | Chemical only | fold 1/5 | SR-ATAD5 | 9/60
FIT  | Chemical only | fold 1/5 | SR-HSE
SAVED | Chemical only | fold 1/5 | SR-HSE | 10/60
FIT

In [ ]:
# Pediatric FAERS download — secure API key and resumable requests

import time
import requests
from google.colab import userdata

# Read the API key from Colab Secrets.
API_KEY = userdata.get("OPENFDA_API_KEY")

if not API_KEY:
    raise RuntimeError(
        "OPENFDA_API_KEY was not found. Add it in Colab Secrets "
        "and enable notebook access."
    )

def fetch_faers_pediatric(limit=1000, max_records=5000):
    records = []
    skip = 0

    base_url = "https://api.fda.gov/drug/event.json"
    query = (
        "(patient.patientonsetageunit:801) AND "
        "(patient.patientonsetage:[0 TO 17])"
    )

    while skip < max_records:
        current_limit = min(limit, max_records - skip)

        params = {
            "search": query,
            "limit": current_limit,
            "skip": skip,
            "api_key": API_KEY,
        }

        try:
            response = requests.get(
                base_url,
                params=params,
                timeout=60,
            )
        except requests.RequestException as error:
            print(f"[skip={skip}] Request error: {error}")
            print("Waiting 30 seconds before retrying...")
            time.sleep(30)
            continue

        if response.status_code == 429:
            print(f"[skip={skip}] Rate limited. Waiting 60 seconds...")
            time.sleep(60)
            continue

        if response.status_code != 200:
            print(
                f"[skip={skip}] Status {response.status_code}: "
                f"{response.text[:300]}"
            )
            break

        data = response.json().get("results", [])

        print(f"[skip={skip}] Downloaded {len(data)} reports")

        if not data:
            break

        records.extend(data)
        skip += len(data)

        if len(data) < current_limit:
            break

        time.sleep(0.3)

    return records


faers_raw = fetch_faers_pediatric(
    limit=1000,
    max_records=5000,
)

print(f"\nPulled {len(faers_raw)} pediatric FAERS reports.")

[skip=0] Downloaded 1000 reports
[skip=1000] Downloaded 1000 reports
[skip=2000] Downloaded 1000 reports
[skip=3000] Downloaded 1000 reports
[skip=4000] Downloaded 1000 reports

Pulled 5000 pediatric FAERS reports.


In [ ]:
# ============================================================
# COMPLETE TOX21 -> PEDIATRIC FAERS TRANSFER ANALYSIS
# One-cell, Google Drive checkpointed version
# ============================================================

%pip install -q rdkit xgboost signaturizer tf-keras joblib

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import re
import json
import time
import hashlib
from pathlib import Path
from urllib.parse import quote

import joblib
import numpy as np
import pandas as pd
import requests

from google.colab import drive, userdata
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.MolStandardize import rdMolStandardize
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    f1_score,
)
from signaturizer import Signaturizer
from xgboost import XGBClassifier

RDLogger.DisableLog("rdApp.*")

# ------------------------------------------------------------
# 1. Configuration and Google Drive
# ------------------------------------------------------------

drive.mount("/content/drive")

SEED = 0
MIN_PED_REPORTS = 5
MAX_FAERS_RECORDS = 5000
BOOTSTRAPS = 2000
PERMUTATION_SEEDS = [101, 202, 303, 404, 505]

SPACES = [
    "B1", "B2", "B4", "B5",
    "C3", "C4", "C5",
    "D1", "D2",
]

TOX21_URL = (
    "https://deepchemdata.s3-us-west-1.amazonaws.com/"
    "datasets/tox21.csv.gz"
)

BASE_DIR = Path(
    "/content/drive/MyDrive/Tox21_multimodal_checkpoints"
)
BASE_DIR.mkdir(parents=True, exist_ok=True)


def atomic_npy(path, array):
    path = Path(path)
    temporary = Path(str(path) + ".temporary.npy")
    np.save(temporary, array)
    os.replace(temporary, path)


def atomic_csv(path, dataframe):
    path = Path(path)
    temporary = Path(str(path) + ".temporary.csv")
    dataframe.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_json(path, value):
    path = Path(path)
    temporary = Path(str(path) + ".temporary.json")

    with open(temporary, "w") as file:
        json.dump(value, file)

    os.replace(temporary, path)


def atomic_joblib(path, value):
    path = Path(path)
    temporary = Path(str(path) + ".temporary.joblib")
    joblib.dump(value, temporary, compress=3)
    os.replace(temporary, path)


print("Drive checkpoint root:", BASE_DIR)

# ------------------------------------------------------------
# 2. Load and reproduce the curated Tox21 dataset
# ------------------------------------------------------------

largest_fragment = rdMolStandardize.LargestFragmentChooser()
uncharger = rdMolStandardize.Uncharger()


def standardize(smiles):
    molecule = Chem.MolFromSmiles(str(smiles))

    if molecule is None:
        return None

    try:
        molecule = rdMolStandardize.Cleanup(molecule)
        molecule = largest_fragment.choose(molecule)
        molecule = uncharger.uncharge(molecule)
        return Chem.MolToSmiles(molecule)
    except Exception:
        return None


raw_tox21 = pd.read_csv(TOX21_URL)

TASKS = [
    column
    for column in raw_tox21.columns
    if column not in ("smiles", "mol_id")
]

raw_tox21["std_smiles"] = raw_tox21["smiles"].map(standardize)
raw_tox21 = raw_tox21.dropna(subset=["std_smiles"])

dedup = (
    raw_tox21.groupby("std_smiles", as_index=False)[TASKS]
    .mean()
)

for task in TASKS:
    values = dedup[task].to_numpy()

    dedup[task] = np.where(
        np.isnan(values),
        np.nan,
        np.where(
            values > 0.5,
            1.0,
            np.where(values < 0.5, 0.0, np.nan),
        ),
    )

dedup = (
    dedup[dedup[TASKS].notna().any(axis=1)]
    .reset_index(drop=True)
)

parsed_molecules = [
    Chem.MolFromSmiles(smiles)
    for smiles in dedup["std_smiles"]
]

keep_indices = [
    index
    for index, molecule in enumerate(parsed_molecules)
    if molecule is not None
]

dedup = dedup.iloc[keep_indices].reset_index(drop=True)
mols = [parsed_molecules[index] for index in keep_indices]

SMILES = dedup["std_smiles"].tolist()
Y = dedup[TASKS].to_numpy(dtype=float)
MASK = ~np.isnan(Y)

DATASET_HASH = hashlib.sha256(
    "\n".join(SMILES).encode("utf-8")
).hexdigest()

SOURCE_DIR = BASE_DIR / DATASET_HASH[:12]
SOURCE_DIR.mkdir(parents=True, exist_ok=True)

print("Tox21 compounds:", len(SMILES))
print("Labels present:", int(MASK.sum()))
print("Dataset hash:", DATASET_HASH)
print("Checkpoint folder:", SOURCE_DIR)

if len(SMILES) != 7587:
    raise RuntimeError(
        f"Expected 7587 curated Tox21 compounds, found {len(SMILES)}."
    )

# ------------------------------------------------------------
# 3. Generate adult chemical features
# ------------------------------------------------------------

fingerprint_generator = (
    rdFingerprintGenerator.GetMorganGenerator(
        radius=2,
        fpSize=2048,
    )
)

X_fp = np.asarray(
    [
        fingerprint_generator.GetFingerprintAsNumPy(molecule)
        for molecule in mols
    ],
    dtype=np.float32,
)

adult_descriptor_frame = pd.DataFrame(
    [Descriptors.CalcMolDescriptors(molecule) for molecule in mols]
)

for column in ("Ipc", "AvgIpc"):
    if column in adult_descriptor_frame.columns:
        adult_descriptor_frame[column] = np.log1p(
            adult_descriptor_frame[column].clip(lower=0)
        )

adult_descriptor_frame = adult_descriptor_frame.replace(
    [np.inf, -np.inf],
    np.nan,
)

descriptor_keep = adult_descriptor_frame.columns[
    adult_descriptor_frame.isna().mean() < 0.05
]

adult_descriptor_frame = adult_descriptor_frame.loc[
    :,
    descriptor_keep,
]

adult_descriptor_medians = adult_descriptor_frame.median()

adult_descriptor_frame = adult_descriptor_frame.fillna(
    adult_descriptor_medians
)

descriptor_columns = adult_descriptor_frame.columns[
    adult_descriptor_frame.std() > 0
].tolist()

adult_descriptor_medians = adult_descriptor_medians.reindex(
    descriptor_columns
)

X_desc = (
    adult_descriptor_frame
    .reindex(columns=descriptor_columns)
    .clip(-1e30, 1e30)
    .to_numpy(dtype=np.float32)
)

X_chem = np.hstack([X_fp, X_desc]).astype(np.float32)

if not np.isfinite(X_chem).all():
    raise RuntimeError("Adult chemical features contain NaN or infinity.")

print("Adult chemical feature matrix:", X_chem.shape)

# ------------------------------------------------------------
# 4. Load or generate adult biological features
# ------------------------------------------------------------

combined_bio_path = SOURCE_DIR / "X_bio_combined.npy"

if combined_bio_path.exists():
    X_bio = np.load(combined_bio_path)
    print("Loaded adult BioFeat matrix:", combined_bio_path)

else:
    print("Adult BioFeat matrix not found.")
    print("Generating Chemical Checker signatures now.")

    adult_bio_blocks = []

    for space in SPACES:
        block_path = SOURCE_DIR / f"X_bio_{space}.npy"

        if block_path.exists():
            block = np.load(block_path)
            print(space, "loaded")

        else:
            print(space, "generating")
            block = (
                Signaturizer(space)
                .predict(SMILES)
                .signature
                .astype(np.float32)
            )

            if block.shape != (len(SMILES), 128):
                raise RuntimeError(
                    f"Unexpected {space} shape: {block.shape}"
                )

            atomic_npy(block_path, block)
            print(space, "saved")

        adult_bio_blocks.append(block)

    X_bio = np.hstack(adult_bio_blocks).astype(np.float32)
    atomic_npy(combined_bio_path, X_bio)

if X_bio.shape != (len(SMILES), len(SPACES) * 128):
    raise RuntimeError(
        f"Unexpected adult BioFeat shape: {X_bio.shape}"
    )

if not np.isfinite(X_bio).all():
    raise RuntimeError("Adult BioFeat contains NaN or infinity.")

print("Adult biological feature matrix:", X_bio.shape)

# ------------------------------------------------------------
# 5. Cache or download pediatric FAERS reports
# ------------------------------------------------------------

PED_DIR = SOURCE_DIR / "pediatric_transfer"
PED_DIR.mkdir(parents=True, exist_ok=True)

RAW_FAERS_PATH = (
    PED_DIR /
    f"faers_pediatric_first_{MAX_FAERS_RECORDS}.json"
)


def get_openfda_key():
    try:
        key = userdata.get("OPENFDA_API_KEY")
        return key.strip() if key else None
    except Exception:
        return None


def fetch_faers_pediatric(
    max_records=MAX_FAERS_RECORDS,
    limit=1000,
):
    records = []
    base_url = "https://api.fda.gov/drug/event.json"

    query = (
        "(patient.patientonsetageunit:801) AND "
        "(patient.patientonsetage:[0 TO 17])"
    )

    api_key = get_openfda_key()

    if not api_key:
        raise RuntimeError(
            "OPENFDA_API_KEY is missing from Colab Secrets."
        )

    for skip in range(0, max_records, limit):
        current_limit = min(limit, max_records - skip)

        parameters = {
            "search": query,
            "limit": current_limit,
            "skip": skip,
            "api_key": api_key,
        }

        for attempt in range(6):
            response = requests.get(
                base_url,
                params=parameters,
                timeout=60,
            )

            if response.status_code == 200:
                break

            if response.status_code in (
                429, 500, 502, 503, 504
            ):
                wait_seconds = min(
                    60,
                    (2 ** attempt) * 5,
                )

                print(
                    f"skip={skip}: HTTP "
                    f"{response.status_code}; "
                    f"waiting {wait_seconds}s"
                )

                time.sleep(wait_seconds)
                continue

            raise RuntimeError(
                f"OpenFDA HTTP {response.status_code}: "
                f"{response.text[:300]}"
            )

        else:
            raise RuntimeError(
                f"OpenFDA repeatedly failed at skip={skip}."
            )

        batch = response.json().get("results", [])

        if not batch:
            break

        records.extend(batch)
        print(
            f"FAERS downloaded: "
            f"{len(records)}/{max_records}"
        )

        if len(batch) < current_limit:
            break

        time.sleep(0.3)

    return records


if RAW_FAERS_PATH.exists():
    with open(RAW_FAERS_PATH) as file:
        faers_raw = json.load(file)

    print(
        "Loaded FAERS reports from Drive:",
        len(faers_raw),
    )

elif (
    "faers_raw" in globals()
    and isinstance(faers_raw, list)
    and len(faers_raw) > 0
):
    faers_raw = faers_raw[:MAX_FAERS_RECORDS]
    atomic_json(RAW_FAERS_PATH, faers_raw)

    print(
        "Saved the reports already in memory to Drive:",
        len(faers_raw),
    )

else:
    faers_raw = fetch_faers_pediatric()
    atomic_json(RAW_FAERS_PATH, faers_raw)

    print(
        "Downloaded and saved FAERS reports:",
        len(faers_raw),
    )

# ------------------------------------------------------------
# 6. Parse suspect drugs and construct pediatric labels
# ------------------------------------------------------------

def latest_report_versions(records):
    latest = {}
    anonymous = []

    for record in records:
        report_id = record.get("safetyreportid")

        if not report_id:
            anonymous.append(record)
            continue

        try:
            version = int(
                record.get("safetyreportversion", 0)
            )
        except Exception:
            version = 0

        previous = latest.get(report_id)

        if previous is None or version > previous[0]:
            latest[report_id] = (version, record)

    return (
        [value[1] for value in latest.values()]
        + anonymous
    )


def parse_suspect_drugs(records):
    rows = []

    for record in latest_report_versions(records):
        serious = int(
            str(record.get("serious", "0")) == "1"
        )

        report_id = record.get("safetyreportid")

        drugs = (
            record.get("patient", {}).get("drug", [])
            or []
        )

        for drug in drugs:
            # Only suspect drugs; exclude concomitant drugs.
            if str(
                drug.get("drugcharacterization", "")
            ) != "1":
                continue

            drug_name = drug.get("medicinalproduct")

            if drug_name:
                rows.append({
                    "report_id": report_id,
                    "drug_name": (
                        drug_name.strip().upper()
                    ),
                    "serious": serious,
                })

    return pd.DataFrame(rows)


mentions_path = (
    PED_DIR / "faers_suspect_drug_mentions.csv"
)

if mentions_path.exists():
    faers_mentions = pd.read_csv(mentions_path)
    print("Loaded parsed FAERS mentions from Drive.")

else:
    faers_mentions = parse_suspect_drugs(
        faers_raw
    )

    if faers_mentions.empty:
        raise RuntimeError(
            "No suspect-drug mentions were extracted."
        )

    atomic_csv(mentions_path, faers_mentions)
    print("Saved parsed FAERS mentions to Drive.")

drug_summary = (
    faers_mentions
    .groupby("drug_name", as_index=False)
    .agg(
        serious_count=("serious", "sum"),
        report_count=("serious", "size"),
    )
)

drug_summary = drug_summary[
    drug_summary["report_count"] >= MIN_PED_REPORTS
].copy()

drug_summary["serious_fraction"] = (
    drug_summary["serious_count"]
    / drug_summary["report_count"]
)

print("Suspect-drug mentions:", len(faers_mentions))

print(
    f"Drug names with at least "
    f"{MIN_PED_REPORTS} reports:",
    len(drug_summary),
)

# ------------------------------------------------------------
# 7. Map pediatric drug names to PubChem structures
# ------------------------------------------------------------

PUBCHEM_CACHE = (
    PED_DIR / "pubchem_name_to_smiles.json"
)

if PUBCHEM_CACHE.exists():
    with open(PUBCHEM_CACHE) as file:
        smiles_map = json.load(file)
else:
    smiles_map = {}


def clean_drug_name(name):
    name = re.sub(
        r"\d+(\.\d+)?\s*(MG|MCG|G|ML|%)\b",
        "",
        name,
        flags=re.I,
    )

    name = re.sub(
        r"\b(TABLET|CAPSULE|INJECTION|SOLUTION|"
        r"ORAL|CHEWABLE|DR|ER|XR|AQ)\b",
        "",
        name,
        flags=re.I,
    )

    return re.sub(r"\s+", " ", name).strip()


def pubchem_lookup(name):
    endpoint = (
        "https://pubchem.ncbi.nlm.nih.gov/rest/"
        "pug/compound/name/"
        f"{quote(name)}/property/"
        "IsomericSMILES,ConnectivitySMILES/JSON"
    )

    try:
        response = requests.get(
            endpoint,
            timeout=30,
        )

        if response.status_code != 200:
            return None

        properties = response.json()[
            "PropertyTable"
        ]["Properties"][0]

        return (
            properties.get("IsomericSMILES")
            or properties.get("ConnectivitySMILES")
        )

    except Exception:
        return None


drug_names = drug_summary["drug_name"].tolist()

for name_index, name in enumerate(
    drug_names,
    start=1,
):
    if name in smiles_map:
        continue

    candidate = pubchem_lookup(name)

    if candidate is None:
        cleaned_name = clean_drug_name(name)

        if cleaned_name and cleaned_name != name:
            candidate = pubchem_lookup(cleaned_name)

    # Do not arbitrarily select one component of a mixture.
    if candidate and "." in candidate:
        candidate = None

    smiles_map[name] = candidate
    atomic_json(PUBCHEM_CACHE, smiles_map)

    if name_index % 25 == 0:
        print(
            f"PubChem: processed "
            f"{name_index}/{len(drug_names)} names"
        )

    time.sleep(0.15)


drug_summary["smiles"] = (
    drug_summary["drug_name"].map(smiles_map)
)

drug_summary["std_smiles"] = (
    drug_summary["smiles"].map(standardize)
)

mapped = drug_summary.dropna(
    subset=["std_smiles"]
).copy()

# Collapse aliases mapping to the same molecule.
pediatric_all = (
    mapped
    .groupby("std_smiles", as_index=False)
    .agg(
        serious_count=("serious_count", "sum"),
        report_count=("report_count", "sum"),
        aliases=(
            "drug_name",
            lambda values: " | ".join(
                sorted(set(values))
            ),
        ),
    )
)

pediatric_all["serious_fraction"] = (
    pediatric_all["serious_count"]
    / pediatric_all["report_count"]
)

pediatric_all["pediatric_serious_majority"] = (
    pediatric_all["serious_fraction"] >= 0.5
).astype(int)

# Preserve the shared-compound transfer design.
source_smiles_set = set(SMILES)

ped = pediatric_all[
    pediatric_all["std_smiles"].isin(
        source_smiles_set
    )
].reset_index(drop=True)

if len(ped) == 0:
    raise RuntimeError(
        "No pediatric structures overlap Tox21."
    )

if ped["pediatric_serious_majority"].nunique() < 2:
    raise RuntimeError(
        "The pediatric cohort contains only one class."
    )

PED_HASH = hashlib.sha256(
    "\n".join(
        ped["std_smiles"].tolist()
    ).encode("utf-8")
).hexdigest()

TRANSFER_DIR = (
    PED_DIR / f"cohort_{PED_HASH[:12]}"
)

TRANSFER_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

atomic_csv(
    TRANSFER_DIR /
    "pediatric_overlap_cohort.csv",
    ped,
)

print(
    "Mapped standardized pediatric structures:",
    len(pediatric_all),
)

print(
    "Shared Tox21/pediatric structures:",
    len(ped),
)

print(
    "Pediatric positive prevalence:",
    round(
        ped[
            "pediatric_serious_majority"
        ].mean(),
        4,
    ),
)

print("Transfer output folder:", TRANSFER_DIR)

# ------------------------------------------------------------
# 8. Generate pediatric chemical and BioFeat features
# ------------------------------------------------------------

pediatric_molecules = [
    Chem.MolFromSmiles(smiles)
    for smiles in ped["std_smiles"]
]

X_pediatric_fp = np.asarray(
    [
        fingerprint_generator
        .GetFingerprintAsNumPy(molecule)
        for molecule in pediatric_molecules
    ],
    dtype=np.float32,
)

pediatric_descriptors = pd.DataFrame(
    [
        Descriptors.CalcMolDescriptors(molecule)
        for molecule in pediatric_molecules
    ]
)

for column in ("Ipc", "AvgIpc"):
    if column in pediatric_descriptors.columns:
        pediatric_descriptors[column] = np.log1p(
            pediatric_descriptors[column].clip(
                lower=0
            )
        )

pediatric_descriptors = (
    pediatric_descriptors.replace(
        [np.inf, -np.inf],
        np.nan,
    )
)

pediatric_descriptors = (
    pediatric_descriptors
    .reindex(columns=descriptor_columns)
    .fillna(adult_descriptor_medians)
)

X_pediatric_desc = (
    pediatric_descriptors
    .clip(-1e30, 1e30)
    .to_numpy(dtype=np.float32)
)

X_pediatric_chem = np.hstack([
    X_pediatric_fp,
    X_pediatric_desc,
]).astype(np.float32)

pediatric_bio_blocks = []

for space in SPACES:
    block_path = (
        TRANSFER_DIR /
        f"pediatric_chemical_checker_{space}.npy"
    )

    if block_path.exists():
        block = np.load(block_path)
        print(space, "pediatric BioFeat loaded")

    else:
        print(space, "pediatric BioFeat generating")

        block = (
            Signaturizer(space)
            .predict(ped["std_smiles"].tolist())
            .signature
            .astype(np.float32)
        )

        expected_shape = (len(ped), 128)

        if block.shape != expected_shape:
            raise RuntimeError(
                f"Invalid {space} block shape: "
                f"{block.shape}"
            )

        if not np.isfinite(block).all():
            raise RuntimeError(
                f"{space} contains NaN or infinity."
            )

        atomic_npy(block_path, block)
        print(space, "pediatric BioFeat saved")

    pediatric_bio_blocks.append(block)

X_pediatric_bio = np.hstack(
    pediatric_bio_blocks
).astype(np.float32)

y_pediatric = ped[
    "pediatric_serious_majority"
].to_numpy(dtype=int)

if not np.isfinite(X_pediatric_chem).all():
    raise RuntimeError(
        "Pediatric chemical features are invalid."
    )

if not np.isfinite(X_pediatric_bio).all():
    raise RuntimeError(
        "Pediatric biological features are invalid."
    )

print(
    "Pediatric chemical feature matrix:",
    X_pediatric_chem.shape,
)

print(
    "Pediatric biological feature matrix:",
    X_pediatric_bio.shape,
)

# ------------------------------------------------------------
# 9. Exclude pediatric compounds from adult training
# ------------------------------------------------------------

pediatric_smiles_set = set(
    ped["std_smiles"]
)

source_train_indices = np.asarray(
    [
        index
        for index, smiles in enumerate(SMILES)
        if smiles not in pediatric_smiles_set
    ],
    dtype=int,
)

remaining_overlap = (
    pediatric_smiles_set.intersection(
        {
            SMILES[index]
            for index in source_train_indices
        }
    )
)

if remaining_overlap:
    raise RuntimeError(
        "Pediatric structures remain in adult training."
    )


def representation_matrices(key):
    if key == "chem":
        return X_chem, X_pediatric_chem

    if key == "chem_true_bio":
        return (
            np.hstack([
                X_chem,
                X_bio,
            ]).astype(np.float32),
            np.hstack([
                X_pediatric_chem,
                X_pediatric_bio,
            ]).astype(np.float32),
        )

    if key == "bio_only":
        return X_bio, X_pediatric_bio

    prefix = "chem_permuted_bio_seed_"

    if key.startswith(prefix):
        seed = int(key[len(prefix):])

        adult_permutation = (
            np.random.default_rng(seed)
            .permutation(len(X_bio))
        )

        pediatric_permutation = (
            np.random.default_rng(seed + 1_000_000)
            .permutation(len(X_pediatric_bio))
        )

        return (
            np.hstack([
                X_chem,
                X_bio[adult_permutation],
            ]).astype(np.float32),
            np.hstack([
                X_pediatric_chem,
                X_pediatric_bio[
                    pediatric_permutation
                ],
            ]).astype(np.float32),
        )

    raise KeyError(key)


REPRESENTATIONS = [
    "chem",
    "chem_true_bio",
    "bio_only",
] + [
    f"chem_permuted_bio_seed_{seed}"
    for seed in PERMUTATION_SEEDS
]

print(
    "Adult training compounds after exclusion:",
    len(source_train_indices),
)

print("Representations:", REPRESENTATIONS)

# ------------------------------------------------------------
# 10. Train models and save after every task
# ------------------------------------------------------------

MODEL_DIR = TRANSFER_DIR / "models"
PREDICTION_DIR = TRANSFER_DIR / "predictions"

MODEL_DIR.mkdir(exist_ok=True)
PREDICTION_DIR.mkdir(exist_ok=True)


def make_xgb(y_train):
    positives = float(y_train.sum())
    negatives = float(len(y_train) - positives)

    return XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.6,
        scale_pos_weight=(
            negatives / max(positives, 1.0)
        ),
        tree_method="hist",
        eval_metric="logloss",
        n_jobs=-1,
        random_state=SEED,
    )


def fit_predict_representation(key):
    source_features, pediatric_features = (
        representation_matrices(key)
    )

    prediction_path = (
        PREDICTION_DIR /
        f"pediatric_task_predictions_{key}.npy"
    )

    if prediction_path.exists():
        predictions = np.load(prediction_path)

        expected_shape = (
            len(ped),
            len(TASKS),
        )

        if predictions.shape != expected_shape:
            raise RuntimeError(
                f"Invalid checkpoint for {key}: "
                f"{predictions.shape}"
            )

    else:
        predictions = np.full(
            (len(ped), len(TASKS)),
            np.nan,
            dtype=np.float32,
        )

        atomic_npy(
            prediction_path,
            predictions,
        )

    for task_index, task in enumerate(TASKS):
        if np.isfinite(
            predictions[:, task_index]
        ).all():
            print("SKIP:", key, task)
            continue

        labeled_train = source_train_indices[
            MASK[source_train_indices, task_index]
        ]

        y_train = Y[
            labeled_train,
            task_index,
        ]

        if len(np.unique(y_train)) < 2:
            raise RuntimeError(
                f"Only one adult class for {task}."
            )

        safe_task = re.sub(
            r"[^A-Za-z0-9_.-]",
            "_",
            task,
        )

        model_path = (
            MODEL_DIR /
            f"{key}__{safe_task}.joblib"
        )

        if model_path.exists():
            model = joblib.load(model_path)
            print("MODEL LOADED:", key, task)

        else:
            print("MODEL FIT:", key, task)

            model = make_xgb(y_train)

            model.fit(
                source_features[labeled_train],
                y_train,
            )

            atomic_joblib(model_path, model)
            print("MODEL SAVED:", key, task)

        predictions[:, task_index] = (
            model.predict_proba(
                pediatric_features
            )[:, 1]
        )

        atomic_npy(
            prediction_path,
            predictions,
        )

        print("PREDICTION SAVED:", key, task)

    del source_features
    del pediatric_features

    return predictions


pediatric_task_predictions = {}

for representation in REPRESENTATIONS:
    pediatric_task_predictions[
        representation
    ] = fit_predict_representation(
        representation
    )

print(
    "All adult models and pediatric predictions complete."
)

# ------------------------------------------------------------
# 11. Calculate pediatric metrics
# ------------------------------------------------------------

# The adult OOF files are used when available. If they are
# missing, pediatric metrics still run, but source transfer-gap
# columns will be NaN.

def load_source_macro_metrics(key):
    oof_path = SOURCE_DIR / f"oof_{key}.npy"

    if not oof_path.exists():
        print(
            "WARNING: missing adult OOF file:",
            oof_path,
        )

        return np.nan, np.nan, None

    oof = np.load(oof_path)
    roc_values = []
    pr_values = []

    for task_index in range(len(TASKS)):
        valid = (
            MASK[:, task_index]
            & np.isfinite(oof[:, task_index])
        )

        labels = Y[valid, task_index]
        scores = oof[valid, task_index]

        if len(np.unique(labels)) < 2:
            continue

        roc_values.append(
            roc_auc_score(labels, scores)
        )

        pr_values.append(
            average_precision_score(
                labels,
                scores,
            )
        )

    return (
        float(np.mean(roc_values)),
        float(np.mean(pr_values)),
        oof,
    )


metric_rows = []
risk_scores = {}
rank_risk_scores = {}

for representation in REPRESENTATIONS:
    (
        source_roc,
        source_pr,
        source_oof,
    ) = load_source_macro_metrics(
        representation
    )

    task_predictions = (
        pediatric_task_predictions[
            representation
        ]
    )

    # Primary pediatric score: average probability
    # across the 12 Tox21 endpoints.
    risk_score = task_predictions.mean(axis=1)

    risk_scores[representation] = risk_score

    pediatric_roc = roc_auc_score(
        y_pediatric,
        risk_score,
    )

    pediatric_pr = average_precision_score(
        y_pediatric,
        risk_score,
    )

    fixed_prediction = (
        risk_score >= 0.5
    ).astype(int)

    # Rank-normalized sensitivity analysis.
    if source_oof is not None:
        rank_columns = []

        for task_index in range(len(TASKS)):
            reference = np.sort(
                source_oof[
                    np.isfinite(
                        source_oof[:, task_index]
                    ),
                    task_index,
                ]
            )

            rank_column = np.searchsorted(
                reference,
                task_predictions[:, task_index],
                side="right",
            ) / len(reference)

            rank_columns.append(rank_column)

        rank_risk_score = np.column_stack(
            rank_columns
        ).mean(axis=1)

        rank_roc = roc_auc_score(
            y_pediatric,
            rank_risk_score,
        )

        rank_pr = average_precision_score(
            y_pediatric,
            rank_risk_score,
        )

    else:
        rank_risk_score = np.full(
            len(y_pediatric),
            np.nan,
        )

        rank_roc = np.nan
        rank_pr = np.nan

    rank_risk_scores[
        representation
    ] = rank_risk_score

    prevalence = float(y_pediatric.mean())

    metric_rows.append({
        "representation": representation,
        "n_pediatric_compounds": len(
            y_pediatric
        ),
        "pediatric_positive_prevalence": (
            prevalence
        ),
        "source_macro_auroc": source_roc,
        "source_macro_auprc": source_pr,
        "pediatric_auroc": pediatric_roc,
        "pediatric_auprc": pediatric_pr,
        "auprc_lift_over_prevalence": (
            pediatric_pr - prevalence
        ),
        "pediatric_mcc_at_0.5": (
            matthews_corrcoef(
                y_pediatric,
                fixed_prediction,
            )
        ),
        "pediatric_f1_at_0.5": f1_score(
            y_pediatric,
            fixed_prediction,
            zero_division=0,
        ),
        "transfer_gap": (
            source_roc - pediatric_roc
            if np.isfinite(source_roc)
            else np.nan
        ),
        "transfer_retention": (
            pediatric_roc / source_roc
            if np.isfinite(source_roc)
            and source_roc != 0
            else np.nan
        ),
        "rank_normalized_pediatric_auroc": (
            rank_roc
        ),
        "rank_normalized_pediatric_auprc": (
            rank_pr
        ),
    })


metrics_df = pd.DataFrame(metric_rows)

metrics_path = (
    TRANSFER_DIR /
    "transfer_representation_metrics.csv"
)

atomic_csv(metrics_path, metrics_df)

prediction_table = ped[[
    "std_smiles",
    "aliases",
    "serious_count",
    "report_count",
    "serious_fraction",
    "pediatric_serious_majority",
]].copy()

for representation in REPRESENTATIONS:
    prediction_table[
        f"risk__{representation}"
    ] = risk_scores[representation]

    prediction_table[
        f"rank_risk__{representation}"
    ] = rank_risk_scores[representation]

predictions_path = (
    TRANSFER_DIR /
    "pediatric_representation_predictions.csv"
)

atomic_csv(
    predictions_path,
    prediction_table,
)

print("\nPEDIATRIC TRANSFER RESULTS")
print("=" * 100)
print(metrics_df.to_string(index=False))

# ------------------------------------------------------------
# 12. Paired bootstrap comparisons
# ------------------------------------------------------------

BOOTSTRAP_PATH = (
    TRANSFER_DIR /
    "paired_bootstrap_representation_comparisons.csv"
)


def paired_bootstrap(
    labels,
    score_a,
    score_b,
    metric,
    n_boot=BOOTSTRAPS,
    seed=0,
):
    if metric == "AUROC":
        metric_function = roc_auc_score

    elif metric == "AUPRC":
        metric_function = (
            average_precision_score
        )

    else:
        raise ValueError(metric)

    point_difference = (
        metric_function(labels, score_b)
        - metric_function(labels, score_a)
    )

    random_generator = (
        np.random.default_rng(seed)
    )

    original_indices = np.arange(
        len(labels)
    )

    differences = []

    while len(differences) < n_boot:
        sampled_indices = (
            random_generator.choice(
                original_indices,
                size=len(original_indices),
                replace=True,
            )
        )

        sampled_labels = labels[
            sampled_indices
        ]

        if np.unique(sampled_labels).size < 2:
            continue

        difference = (
            metric_function(
                sampled_labels,
                score_b[sampled_indices],
            )
            - metric_function(
                sampled_labels,
                score_a[sampled_indices],
            )
        )

        differences.append(difference)

    differences = np.asarray(differences)

    ci_low, ci_high = np.percentile(
        differences,
        [2.5, 97.5],
    )

    probability_lower = (
        np.sum(differences <= 0) + 1
    ) / (len(differences) + 1)

    probability_upper = (
        np.sum(differences >= 0) + 1
    ) / (len(differences) + 1)

    p_value = min(
        1.0,
        2 * min(
            probability_lower,
            probability_upper,
        ),
    )

    return (
        point_difference,
        ci_low,
        ci_high,
        p_value,
    )


if BOOTSTRAP_PATH.exists():
    bootstrap_df = pd.read_csv(
        BOOTSTRAP_PATH
    )
else:
    bootstrap_df = pd.DataFrame()


comparison_pairs = [
    ("chem", "chem_true_bio")
]

comparison_pairs.extend([
    (
        f"chem_permuted_bio_seed_{seed}",
        "chem_true_bio",
    )
    for seed in PERMUTATION_SEEDS
])

existing_comparisons = set()

if not bootstrap_df.empty:
    existing_comparisons = set(
        zip(
            bootstrap_df["model_a"],
            bootstrap_df["model_b"],
            bootstrap_df["metric"],
        )
    )

bootstrap_rows = (
    bootstrap_df.to_dict("records")
    if not bootstrap_df.empty
    else []
)

for pair_index, (
    model_a,
    model_b,
) in enumerate(comparison_pairs):

    for metric_index, metric in enumerate(
        ("AUROC", "AUPRC")
    ):
        comparison_key = (
            model_a,
            model_b,
            metric,
        )

        if comparison_key in existing_comparisons:
            print(
                "BOOTSTRAP SKIP:",
                model_a,
                model_b,
                metric,
            )
            continue

        (
            difference,
            ci_low,
            ci_high,
            p_value,
        ) = paired_bootstrap(
            y_pediatric,
            risk_scores[model_a],
            risk_scores[model_b],
            metric,
            seed=(
                SEED
                + pair_index * 10
                + metric_index
            ),
        )

        bootstrap_rows.append({
            "model_a": model_a,
            "model_b": model_b,
            "metric": metric,
            "difference_b_minus_a": difference,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "p_boot_two_sided": p_value,
            "n_boot": BOOTSTRAPS,
        })

        bootstrap_df = pd.DataFrame(
            bootstrap_rows
        )

        atomic_csv(
            BOOTSTRAP_PATH,
            bootstrap_df,
        )

        print(
            "BOOTSTRAP SAVED:",
            model_a,
            model_b,
            metric,
        )


print("\nPAIRED BOOTSTRAP RESULTS")
print("=" * 100)
print(bootstrap_df.to_string(index=False))

print("\nANALYSIS COMPLETE")
print("=" * 100)
print("Results folder:")
print(TRANSFER_DIR)

print("\nMain results:")
print(metrics_path)

print("\nBootstrap comparisons:")
print(BOOTSTRAP_PATH)

print("\nCompound-level predictions:")
print(predictions_path)

Mounted at /content/drive
Drive checkpoint root: /content/drive/MyDrive/Tox21_multimodal_checkpoints
Tox21 compounds: 7587
Labels present: 75521
Dataset hash: 27f4dc5e18e00f6e10d83fce74026d7cdef618851383db45bfcf67042f03aaaa
Checkpoint folder: /content/drive/MyDrive/Tox21_multimodal_checkpoints/27f4dc5e18e0
Adult chemical feature matrix: (7587, 2262)
Loaded adult BioFeat matrix: /content/drive/MyDrive/Tox21_multimodal_checkpoints/27f4dc5e18e0/X_bio_combined.npy
Adult biological feature matrix: (7587, 1152)
Saved the reports already in memory to Drive: 5000
Saved parsed FAERS mentions to Drive.
Suspect-drug mentions: 9634
Drug names with at least 5 reports: 368
PubChem: processed 25/368 names
PubChem: processed 50/368 names
PubChem: processed 75/368 names
PubChem: processed 100/368 names
PubChem: processed 125/368 names
PubChem: processed 150/368 names
PubChem: processed 175/368 names
PubChem: processed 200/368 names
PubChem: processed 225/368 names
PubChem: processed 250/368 names
PubCh

Parsing SMILES: 46it [00:00, 3674.94it/s]
Generating signatures:   0%|          | 0/1 [00:00<?, ?it/s]

1/1 [==============================] - 0s 116ms/step


Generating signatures: 100%|██████████| 1/1 [00:00<00:00,  1.38it/s]

B1 pediatric BioFeat saved
B2 pediatric BioFeat generating


...


Parsing SMILES: 46it [00:00, 3543.59it/s]
Generating signatures:   0%|          | 0/1 [00:00<?, ?it/s]

1/1 [==============================] - 0s 99ms/step


Generating signatures: 100%|██████████| 1/1 [00:00<00:00,  2.13it/s]

D2 pediatric BioFeat saved
Pediatric chemical feature matrix: (46, 2262)
Pediatric biological feature matrix: (46, 1152)
Adult training compounds after exclusion: 7541
Representations: ['chem', 'chem_true_bio', 'bio_only', 'chem_permuted_bio_seed_101', 'chem_permuted_bio_seed_202', 'chem_permuted_bio_seed_303', 'chem_permuted_bio_seed_404', 'chem_permuted_bio_seed_505']
MODEL FIT: chem NR-AR


MODEL SAVED: chem NR-AR
PREDICTION SAVED: chem NR-AR
MODEL FIT: chem NR-AR-LBD
MODEL SAVED: chem NR-AR-LBD
PREDICTION SAVED: chem NR-AR-LBD
MODEL FIT: chem NR-AhR
MODEL SAVED: chem NR-AhR
PREDICTION SAVED: chem NR-AhR
MODEL FIT: chem NR-Aromatase
MODEL SAVED: chem NR-Aromatase
PREDICTION SAVED: chem NR-Aromatase
MODEL FIT: chem NR-ER
MODEL SAVED: chem NR-ER
PREDICTION SAVED: chem NR-ER
MODEL FIT: chem NR-ER-LBD
MODEL SAVED: chem NR-ER-LBD
PREDICTION SAVED: chem NR-ER-LBD
MODEL FIT: chem NR-PPAR-gamma
MODEL SAVED: chem NR-PPAR-gamma
PREDICTION SAVED: chem NR-PPAR-gamma
MODEL FIT: chem SR-ARE
MODEL SAVED: chem SR-ARE
PREDICTION SAVED: chem SR-ARE
MODEL FIT: chem SR-ATAD5
MODEL SAVED: chem SR-ATAD5
PREDICTION SAVED: chem SR-ATAD5
MODEL FIT: chem SR-HSE
MODEL SAVED: chem SR-HSE
PREDICTION SAVED: chem SR-HSE
MODEL FIT: chem SR-MMP
MODEL SAVED: chem SR-MMP
PREDICTION SAVED: chem SR-MMP
MODEL FIT: chem SR-p53
MODEL SAVED: chem SR-p53
PREDICTION SAVED: chem SR-p53
MODEL FIT: chem_true_bio NR-A

In [ ]:
# Paste this entire script into ONE Google Colab cell.

# =============================================================================
# PEDIATRIC MULTISOURCE CLINICAL-SAFETY PIPELINE
#
# Primary source:
#   ClinicalTrials.gov pediatric-only interventional studies with results
#
# External evidence:
#   Pediatric versus adult FAERS reports
#
# Optional external source:
#   Canada Vigilance CSV placed at:
#   /content/drive/MyDrive/pediatric_clinical_safety_pipeline/raw/
#       canada_vigilance.csv
#
# All downloads, mappings, BioFeat arrays, models, predictions, and results are
# saved to Google Drive. Rerunning the cell resumes previously completed work.
# =============================================================================

import os
import sys
import subprocess

os.environ["TF_USE_LEGACY_KERAS"] = "1"

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "rdkit",
        "xgboost",
        "signaturizer",
        "tf-keras",
        "joblib",
        "scipy",
    ],
    check=True,
)

import re
import json
import time
import hashlib
import warnings
from pathlib import Path
from urllib.parse import quote

import joblib
import numpy as np
import pandas as pd
import requests

from scipy.stats import spearmanr
from google.colab import drive, userdata

from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    mean_absolute_error,
    mean_squared_error,
)
from sklearn.model_selection import GroupKFold

from xgboost import XGBClassifier, XGBRegressor
from signaturizer import Signaturizer


# =============================================================================
# CONFIGURATION
# =============================================================================

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore", category=FutureWarning)

drive.mount("/content/drive")

SEED = 20260915

MAX_CT_STUDIES = 10_000
FAERS_PER_DOMAIN = 25_000

MIN_TRIAL_AT_RISK = 30
MIN_ENDPOINT_STRUCTURES = 80
MIN_BINARY_CLASS = 20

N_FOLDS = 5
N_BOOT = 2000

SPACES = [
    "B1",
    "B2",
    "B4",
    "B5",
    "C3",
    "C4",
    "C5",
    "D1",
    "D2",
]

ROOT = Path(
    "/content/drive/MyDrive/pediatric_clinical_safety_pipeline"
)

RAW = ROOT / "raw"
CACHE = ROOT / "cache"
OUT = ROOT / "outputs"
MODELS = ROOT / "models"

for folder in [RAW, CACHE, OUT, MODELS]:
    folder.mkdir(parents=True, exist_ok=True)

CANADA_FILE = RAW / "canada_vigilance.csv"


# =============================================================================
# SAFE SAVING AND REQUEST HELPERS
# =============================================================================

def atomic_csv(path, frame):
    path = Path(path)
    temporary = Path(str(path) + ".temporary.csv")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_json(path, value):
    path = Path(path)
    temporary = Path(str(path) + ".temporary.json")

    with open(temporary, "w") as file:
        json.dump(value, file)

    os.replace(temporary, path)


def atomic_npy(path, value):
    path = Path(path)
    temporary = Path(str(path) + ".temporary.npy")

    np.save(temporary, value)
    os.replace(temporary, path)


def atomic_joblib(path, value):
    path = Path(path)
    temporary = Path(str(path) + ".temporary.joblib")

    joblib.dump(value, temporary, compress=3)
    os.replace(temporary, path)


def safe_get(
    url,
    *,
    params=None,
    attempts=7,
    timeout=90,
):
    for attempt in range(attempts):
        try:
            response = requests.get(
                url,
                params=params,
                timeout=timeout,
            )
        except requests.RequestException as error:
            response = None
            print("Request error:", error)

        if response is not None and response.status_code == 200:
            return response

        status = (
            None if response is None
            else response.status_code
        )

        if status not in [
            None,
            429,
            500,
            502,
            503,
            504,
        ]:
            raise RuntimeError(
                f"HTTP {status}: {response.text[:500]}"
            )

        wait_seconds = min(
            60,
            3 * (2 ** attempt),
        )

        print(
            f"HTTP {status}; retrying in "
            f"{wait_seconds} seconds"
        )

        time.sleep(wait_seconds)

    raise RuntimeError(
        f"Repeated request failure: {url}"
    )


# Save a copy of the exact running cell to Drive.

try:
    current_code = (
        get_ipython()
        .history_manager
        .input_hist_raw[-1]
    )

    code_path = (
        ROOT /
        "pediatric_clinical_multisource_pipeline.py"
    )

    code_path.write_text(current_code)

    print("Saved pipeline code to:", code_path)

except Exception as error:
    print(
        "Could not automatically save cell source:",
        error,
    )


# =============================================================================
# OUTCOME HARMONIZATION
# =============================================================================

OUTCOME_PATTERNS = {
    "CARDIAC": (
        r"cardiac|heart|arrhythm|tachy|brady|"
        r"myocard|cardio"
    ),
    "HEPATIC": (
        r"hepat|liver|bilirubin|transamin|"
        r"alanine amino|aspartate amino"
    ),
    "RENAL": (
        r"renal|kidney|neph|creatinine|urinary"
    ),
    "NEUROLOGIC": (
        r"nervous|neurolog|seizure|convulsion|"
        r"headache|dizz|tremor"
    ),
    "RESPIRATORY": (
        r"respirat|pulmon|dyspn|hypoxi|"
        r"bronch|pneum"
    ),
    "HEMATOLOGIC": (
        r"blood|hemat|anaemi|anemi|thromb|"
        r"neutrop|leukop|coagul"
    ),
    "GASTROINTESTINAL": (
        r"gastro|intestinal|nausea|vomit|"
        r"diarr|abdominal|pancrea"
    ),
    "DERMATOLOGIC": (
        r"skin|subcutaneous|rash|urtic|"
        r"dermat|stevens.johnson"
    ),
    "IMMUNE_INFECTION": (
        r"immune|infection|infectious|sepsis|"
        r"anaphyl|hypersens"
    ),
    "ENDOCRINE_METABOLIC": (
        r"endocr|metabol|glucose|glyc|"
        r"thyroid|electrolyte"
    ),
    "PSYCHIATRIC": (
        r"psychiatr|suicid|depress|anxiety|"
        r"behavio|sleep"
    ),
    "CONGENITAL_DEVELOPMENTAL": (
        r"congenital|familial|genetic|"
        r"development|fetal|foetal|birth defect"
    ),
}


def harmonize_outcome(organ_system, term=""):
    text = (
        f"{organ_system or ''} {term or ''}"
        .lower()
    )

    for category, pattern in OUTCOME_PATTERNS.items():
        if re.search(
            pattern,
            text,
            flags=re.IGNORECASE,
        ):
            return category

    return "OTHER"


def norm_text(value):
    return re.sub(
        r"[^a-z0-9]+",
        " ",
        str(value).lower(),
    ).strip()


def age_to_years(text):
    if text is None:
        return None

    match = re.search(
        r"([0-9.]+)\s*(year|month|week|day)",
        str(text),
        re.IGNORECASE,
    )

    if not match:
        return None

    value = float(match.group(1))
    unit = match.group(2).lower()

    if unit.startswith("year"):
        return value

    if unit.startswith("month"):
        return value / 12

    if unit.startswith("week"):
        return value / 52

    return value / 365.25


# =============================================================================
# CLINICALTRIALS.GOV DOWNLOAD
# =============================================================================

CT_CACHE = (
    RAW /
    "clinicaltrials_interventional_with_results_resume.json"
)


def fetch_ctgov():
    url = "https://clinicaltrials.gov/api/v2/studies"

    params = {
        "query.term": (
            "AREA[StdAge]CHILD AND "
            "AREA[StudyType]INTERVENTIONAL AND "
            "AREA[HasResults]true"
        ),
        "pageSize": 100,
        "format": "json",
    }

    if CT_CACHE.exists():
        with open(CT_CACHE) as file:
            checkpoint = json.load(file)

        studies = checkpoint.get("studies", [])
        page_token = checkpoint.get(
            "next_page_token"
        )
        complete = bool(
            checkpoint.get("complete", False)
        )

        print(
            "Resuming ClinicalTrials.gov at "
            f"{len(studies):,} studies"
        )

        if (
            complete
            or len(studies) >= MAX_CT_STUDIES
        ):
            return studies[:MAX_CT_STUDIES]

    else:
        studies = []
        page_token = None

    while len(studies) < MAX_CT_STUDIES:
        if page_token:
            params["pageToken"] = page_token
        elif "pageToken" in params:
            del params["pageToken"]

        payload = safe_get(
            url,
            params=params,
        ).json()

        page_studies = payload.get(
            "studies",
            [],
        )

        studies.extend(page_studies)

        page_token = payload.get(
            "nextPageToken"
        )

        print(
            "ClinicalTrials.gov: "
            f"{len(studies):,} studies"
        )

        atomic_json(
            CT_CACHE,
            {
                "studies": studies,
                "next_page_token": page_token,
                "complete": not bool(page_token),
            },
        )

        if not page_token:
            break

    return studies[:MAX_CT_STUDIES]


ct_studies = fetch_ctgov()

print(
    "ClinicalTrials.gov studies ready:",
    len(ct_studies),
)


# =============================================================================
# CLINICAL TRIAL PARSING
# =============================================================================

CONTROL_WORDS = re.compile(
    (
        r"placebo|vehicle|usual care|standard care|"
        r"no treatment|control"
    ),
    re.IGNORECASE,
)


def pediatric_trial(study):
    eligibility = (
        study
        .get("protocolSection", {})
        .get("eligibilityModule", {})
    )

    minimum = age_to_years(
        eligibility.get("minimumAge")
    )

    maximum = age_to_years(
        eligibility.get("maximumAge")
    )

    # Only pediatric-only trials are accepted.
    # Studies allowing adults are excluded.

    return (
        minimum is not None
        and maximum is not None
        and minimum < 18
        and maximum <= 17.999
    )


def trial_rows(study):
    if not pediatric_trial(study):
        return []

    protocol = study.get(
        "protocolSection",
        {},
    )

    nct_id = (
        protocol
        .get("identificationModule", {})
        .get("nctId")
    )

    arms_module = protocol.get(
        "armsInterventionsModule",
        {},
    )

    interventions = (
        arms_module.get("interventions", [])
        or []
    )

    arms = (
        arms_module.get("armGroups", [])
        or []
    )

    adverse = (
        study
        .get("resultsSection", {})
        .get("adverseEventsModule", {})
    )

    event_groups = (
        adverse.get("eventGroups", [])
        or []
    )

    if not event_groups:
        return []

    # Map each arm to exactly one non-control
    # drug intervention.

    label_to_drugs = {}

    for intervention in interventions:
        if intervention.get("type") != "DRUG":
            continue

        name = intervention.get(
            "name",
            "",
        ).strip()

        if (
            not name
            or CONTROL_WORDS.search(name)
        ):
            continue

        arm_labels = (
            intervention.get(
                "armGroupLabels",
                [],
            )
            or []
        )

        for arm_label in arm_labels:
            normalized_label = norm_text(
                arm_label
            )

            label_to_drugs.setdefault(
                normalized_label,
                set(),
            ).add(name)

    clean_arm_drug = {}

    for arm in arms:
        label = norm_text(
            arm.get("label", "")
        )

        drugs = label_to_drugs.get(
            label,
            set(),
        )

        if (
            len(drugs) == 1
            and not CONTROL_WORDS.search(
                arm.get("label", "")
            )
        ):
            clean_arm_drug[label] = next(
                iter(drugs)
            )

    group_to_drug = {}

    for group in event_groups:
        label = norm_text(
            group.get("title", "")
        )

        drug = clean_arm_drug.get(label)

        if drug:
            group_to_drug[
                group.get("id")
            ] = drug

    rows = []

    # Overall adverse-event outcomes.

    for group in event_groups:
        group_id = group.get("id")
        drug = group_to_drug.get(group_id)

        if not drug:
            continue

        total_outcomes = [
            (
                "ANY_SERIOUS",
                "seriousNumAffected",
                "seriousNumAtRisk",
            ),
            (
                "MORTALITY",
                "deathsNumAffected",
                "deathsNumAtRisk",
            ),
            (
                "ANY_OTHER_AE",
                "otherNumAffected",
                "otherNumAtRisk",
            ),
        ]

        for (
            outcome,
            affected_key,
            risk_key,
        ) in total_outcomes:

            affected = group.get(affected_key)
            at_risk = group.get(risk_key)

            if (
                affected is not None
                and at_risk
            ):
                rows.append(
                    {
                        "source": "CLINICALTRIALS",
                        "study_id": nct_id,
                        "drug_name": drug,
                        "outcome": outcome,
                        "event_term": outcome,
                        "serious": (
                            outcome == "ANY_SERIOUS"
                        ),
                        "affected": int(affected),
                        "at_risk": int(at_risk),
                        "aggregation": "arm_total",
                    }
                )

    # Organ-system adverse events.

    detailed = []

    for seriousness, event_key in [
        (True, "seriousEvents"),
        (False, "otherEvents"),
    ]:
        events = (
            adverse.get(event_key, [])
            or []
        )

        for event in events:
            category = harmonize_outcome(
                event.get("organSystem"),
                event.get("term"),
            )

            if category == "OTHER":
                continue

            statistics = (
                event.get("stats", [])
                or []
            )

            for statistic in statistics:
                group_id = statistic.get(
                    "groupId"
                )

                drug = group_to_drug.get(
                    group_id
                )

                affected = statistic.get(
                    "numAffected"
                )

                at_risk = statistic.get(
                    "numAtRisk"
                )

                if (
                    drug
                    and affected is not None
                    and at_risk
                ):
                    detailed.append(
                        {
                            "source": "CLINICALTRIALS",
                            "study_id": nct_id,
                            "drug_name": drug,
                            "outcome": category,
                            "event_term": event.get(
                                "term"
                            ),
                            "serious": seriousness,
                            "affected": int(
                                affected
                            ),
                            "at_risk": int(
                                at_risk
                            ),
                            "aggregation": (
                                "max_within_"
                                "study_arm_category"
                            ),
                        }
                    )

    if detailed:
        detail_df = pd.DataFrame(detailed)

        # Avoid summing event terms because one
        # participant can experience multiple terms.
        # Retain the largest affected count for each
        # study, drug arm, and category.

        detail_df = (
            detail_df
            .sort_values("affected")
            .groupby(
                [
                    "source",
                    "study_id",
                    "drug_name",
                    "outcome",
                    "aggregation",
                ],
                as_index=False,
            )
            .tail(1)
        )

        rows.extend(
            detail_df.to_dict("records")
        )

    return rows


CT_ROWS_PATH = (
    CACHE /
    "clinicaltrials_harmonized_event_rows.csv"
)

if CT_ROWS_PATH.exists():
    ct_rows = pd.read_csv(CT_ROWS_PATH)

    print(
        "Loaded harmonized trial rows:",
        len(ct_rows),
    )

else:
    all_trial_rows = []

    for study_number, study in enumerate(
        ct_studies,
        start=1,
    ):
        all_trial_rows.extend(
            trial_rows(study)
        )

        if study_number % 500 == 0:
            print(
                "Parsed clinical trial",
                study_number,
            )

    ct_rows = pd.DataFrame(all_trial_rows)

    if ct_rows.empty:
        raise RuntimeError(
            "No eligible pediatric trial arms "
            "were parsed."
        )

    atomic_csv(
        CT_ROWS_PATH,
        ct_rows,
    )

print(
    "Clinical-trial event rows:",
    len(ct_rows),
)

print(
    "Clinical-trial unique drug names:",
    ct_rows["drug_name"].nunique(),
)


# =============================================================================
# OPENFDA KEY
# =============================================================================

try:
    OPENFDA_KEY = (
        userdata
        .get("OPENFDA_API_KEY")
        .strip()
    )

except Exception:
    OPENFDA_KEY = ""

if not OPENFDA_KEY:
    raise RuntimeError(
        "Add OPENFDA_API_KEY to Colab Secrets "
        "and enable notebook access."
    )


# =============================================================================
# FAERS DOWNLOAD
# =============================================================================

def fetch_faers(domain):
    output_path = (
        RAW /
        f"faers_{domain}_{FAERS_PER_DOMAIN}.json"
    )

    if output_path.exists():
        with open(output_path) as file:
            records = json.load(file)

        print(
            f"Resuming {domain} FAERS at "
            f"{len(records):,} reports"
        )

    else:
        records = []

    if domain == "pediatric":
        query = (
            "(patient.patientonsetageunit:801) "
            "AND "
            "(patient.patientonsetage:[0 TO 17])"
        )

    elif domain == "adult":
        query = (
            "(patient.patientonsetageunit:801) "
            "AND "
            "(patient.patientonsetage:[18 TO 100])"
        )

    else:
        raise ValueError(
            "Domain must be pediatric or adult."
        )

    for skip in range(
        len(records),
        FAERS_PER_DOMAIN,
        1000,
    ):
        current_limit = min(
            1000,
            FAERS_PER_DOMAIN - skip,
        )

        params = {
            "search": query,
            "limit": current_limit,
            "skip": skip,
            "api_key": OPENFDA_KEY,
        }

        payload = safe_get(
            "https://api.fda.gov/drug/event.json",
            params=params,
        ).json()

        batch = payload.get(
            "results",
            [],
        )

        if not batch:
            break

        records.extend(batch)

        print(
            f"{domain} FAERS: "
            f"{len(records):,} reports"
        )

        atomic_json(
            output_path,
            records,
        )

        if len(batch) < current_limit:
            break

        time.sleep(0.25)

    atomic_json(
        output_path,
        records,
    )

    return records


def latest_faers(records):
    latest = {}

    for record in records:
        report_id = record.get(
            "safetyreportid"
        )

        if not report_id:
            report_id = hashlib.sha1(
                json.dumps(
                    record,
                    sort_keys=True,
                ).encode()
            ).hexdigest()

        try:
            version = int(
                record.get(
                    "safetyreportversion",
                    0,
                )
            )

        except Exception:
            version = 0

        if (
            report_id not in latest
            or version > latest[report_id][0]
        ):
            latest[report_id] = (
                version,
                record,
            )

    return [
        value[1]
        for value in latest.values()
    ]


def parse_faers(records, domain):
    rows = []

    for record in latest_faers(records):
        report_id = record.get(
            "safetyreportid"
        )

        reactions = (
            record
            .get("patient", {})
            .get("reaction", [])
            or []
        )

        categories = {
            harmonize_outcome(
                "",
                reaction.get(
                    "reactionmeddrapt",
                    "",
                ),
            )
            for reaction in reactions
        }

        categories.discard("OTHER")

        drugs = (
            record
            .get("patient", {})
            .get("drug", [])
            or []
        )

        suspect_names = {
            drug.get(
                "medicinalproduct",
                "",
            ).strip().upper()

            for drug in drugs

            if (
                str(
                    drug.get(
                        "drugcharacterization",
                        "",
                    )
                ) == "1"
                and drug.get(
                    "medicinalproduct"
                )
            )
        }

        for drug_name in suspect_names:
            # Keep every suspect-drug report for
            # the ROR denominator.
            rows.append(
                {
                    "source": "FAERS",
                    "domain": domain,
                    "report_id": report_id,
                    "drug_name": drug_name,
                    "outcome": "__ALL_REPORTS__",
                }
            )

            for category in categories:
                rows.append(
                    {
                        "source": "FAERS",
                        "domain": domain,
                        "report_id": report_id,
                        "drug_name": drug_name,
                        "outcome": category,
                    }
                )

    return pd.DataFrame(rows)


FAERS_ROWS_PATH = (
    CACHE /
    "faers_harmonized_event_rows_v2.csv"
)

if FAERS_ROWS_PATH.exists():
    faers_rows = pd.read_csv(
        FAERS_ROWS_PATH
    )

    print(
        "Loaded harmonized FAERS rows:",
        len(faers_rows),
    )

else:
    pediatric_raw = fetch_faers(
        "pediatric"
    )

    adult_raw = fetch_faers(
        "adult"
    )

    pediatric_faers = parse_faers(
        pediatric_raw,
        "pediatric",
    )

    adult_faers = parse_faers(
        adult_raw,
        "adult",
    )

    faers_rows = pd.concat(
        [
            pediatric_faers,
            adult_faers,
        ],
        ignore_index=True,
    )

    atomic_csv(
        FAERS_ROWS_PATH,
        faers_rows,
    )

print(
    "FAERS harmonized rows:",
    len(faers_rows),
)

print(
    "FAERS unique drug names:",
    faers_rows["drug_name"].nunique(),
)


# =============================================================================
# OPTIONAL CANADA VIGILANCE FILE
# =============================================================================

def find_column(columns, candidates):
    normalized = {
        norm_text(column): column
        for column in columns
    }

    for candidate in candidates:
        for normalized_name, original in (
            normalized.items()
        ):
            if candidate in normalized_name:
                return original

    return None


canada_rows = pd.DataFrame(
    columns=[
        "source",
        "domain",
        "report_id",
        "drug_name",
        "outcome",
    ]
)

if CANADA_FILE.exists():
    canada = pd.read_csv(
        CANADA_FILE,
        low_memory=False,
    )

    drug_column = find_column(
        canada.columns,
        [
            "drug name",
            "product name",
            "medicinal product",
        ],
    )

    event_column = find_column(
        canada.columns,
        [
            "reaction term",
            "adverse reaction",
            "meddra",
            "reaction",
        ],
    )

    age_column = find_column(
        canada.columns,
        [
            "age years",
            "patient age",
            "age",
        ],
    )

    report_column = find_column(
        canada.columns,
        [
            "report id",
            "case id",
            "report number",
        ],
    )

    if not all(
        [
            drug_column,
            event_column,
            age_column,
        ]
    ):
        raise RuntimeError(
            "Canada columns were not recognized: "
            f"{list(canada.columns)}"
        )

    numeric_age = pd.to_numeric(
        canada[age_column],
        errors="coerce",
    )

    pediatric_canada = canada[
        numeric_age < 18
    ].copy()

    if report_column:
        report_ids = (
            pediatric_canada[
                report_column
            ].astype(str)
        )
    else:
        report_ids = (
            pediatric_canada
            .index
            .astype(str)
        )

    canada_rows = pd.DataFrame(
        {
            "source": "CANADA_VIGILANCE",
            "domain": "pediatric",
            "report_id": report_ids,
            "drug_name": (
                pediatric_canada[
                    drug_column
                ]
                .astype(str)
                .str.strip()
                .str.upper()
            ),
            "outcome": [
                harmonize_outcome(
                    "",
                    value,
                )
                for value in pediatric_canada[
                    event_column
                ]
            ],
        }
    )

    canada_rows = (
        canada_rows[
            canada_rows["outcome"] != "OTHER"
        ]
        .drop_duplicates()
    )

    atomic_csv(
        CACHE /
        "canada_harmonized_event_rows.csv",
        canada_rows,
    )

    print(
        "Canada harmonized rows:",
        len(canada_rows),
    )

else:
    print(
        "Canada validation skipped. "
        "The optional file was not found:",
        CANADA_FILE,
    )


# =============================================================================
# DRUG NAME TO CHEMICAL STRUCTURE HARMONIZATION
# =============================================================================

PUBCHEM_CACHE = (
    CACHE /
    "pubchem_drug_to_smiles.json"
)

if PUBCHEM_CACHE.exists():
    with open(PUBCHEM_CACHE) as file:
        pubchem_cache = json.load(file)

else:
    pubchem_cache = {}


_largest_fragment = (
    rdMolStandardize
    .LargestFragmentChooser()
)

_uncharger = (
    rdMolStandardize
    .Uncharger()
)


def standardize_smiles(smiles):
    molecule = Chem.MolFromSmiles(
        str(smiles)
    )

    if molecule is None:
        return None

    try:
        molecule = rdMolStandardize.Cleanup(
            molecule
        )

        molecule = _largest_fragment.choose(
            molecule
        )

        molecule = _uncharger.uncharge(
            molecule
        )

        return Chem.MolToSmiles(
            molecule
        )

    except Exception:
        return None


def clean_drug_name(name):
    text = re.sub(
        (
            r"\b(placebo|tablet|capsule|injection|"
            r"solution|oral|iv|intravenous)\b"
        ),
        " ",
        str(name),
        flags=re.IGNORECASE,
    )

    text = re.sub(
        (
            r"\b\d+(\.\d+)?\s*"
            r"(mg|mcg|g|ml|%)\b"
        ),
        " ",
        text,
        flags=re.IGNORECASE,
    )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def pubchem_lookup(name):
    if name in pubchem_cache:
        return pubchem_cache[name]

    result = None

    candidates = list(
        dict.fromkeys(
            [
                str(name).strip(),
                clean_drug_name(name),
            ]
        )
    )

    for candidate in candidates:
        if not candidate:
            continue

        url = (
            "https://pubchem.ncbi.nlm.nih.gov/"
            "rest/pug/compound/name/"
            + quote(candidate)
            + "/property/ConnectivitySMILES/JSON"
        )

        try:
            response = requests.get(
                url,
                timeout=30,
            )

            if response.status_code == 200:
                properties = (
                    response.json()
                    ["PropertyTable"]
                    ["Properties"][0]
                )

                result = properties.get(
                    "ConnectivitySMILES"
                )

                # Mixtures and combinations are rejected.
                if result and "." not in result:
                    break

                result = None

        except Exception:
            pass

        time.sleep(0.1)

    if result:
        result = standardize_smiles(result)

    pubchem_cache[name] = result

    atomic_json(
        PUBCHEM_CACHE,
        pubchem_cache,
    )

    return result


all_names = sorted(
    set(
        ct_rows["drug_name"].dropna()
    )
    | set(
        faers_rows["drug_name"].dropna()
    )
    | set(
        canada_rows["drug_name"].dropna()
    )
)

for name_number, name in enumerate(
    all_names,
    start=1,
):
    pubchem_lookup(name)

    if name_number % 25 == 0:
        print(
            f"PubChem mapping "
            f"{name_number}/{len(all_names)}"
        )


for frame in [
    ct_rows,
    faers_rows,
    canada_rows,
]:
    if len(frame):
        frame["std_smiles"] = (
            frame["drug_name"]
            .map(pubchem_cache)
        )


ct_rows = ct_rows.dropna(subset=["std_smiles"])
faers_rows = faers_rows.dropna(subset=["std_smiles"])

# Canada Vigilance is optional, so create the expected column when skipped.
if "std_smiles" not in canada_rows.columns:
    canada_rows["std_smiles"] = pd.Series(
        index=canada_rows.index,
        dtype="object",
    )

canada_rows = canada_rows.dropna(subset=["std_smiles"])


# =============================================================================
# CLINICAL TRIAL STRUCTURE-LEVEL OUTCOMES
# =============================================================================

ct_structure = (
    ct_rows
    .groupby(
        [
            "std_smiles",
            "outcome",
        ],
        as_index=False,
    )
    .agg(
        affected=(
            "affected",
            "sum",
        ),
        at_risk=(
            "at_risk",
            "sum",
        ),
        n_studies=(
            "study_id",
            "nunique",
        ),
        aliases=(
            "drug_name",
            lambda values: " | ".join(
                sorted(set(values))
            ),
        ),
    )
)

ct_structure = ct_structure[
    ct_structure["at_risk"]
    >= MIN_TRIAL_AT_RISK
].copy()

ct_structure["event_rate_smoothed"] = (
    ct_structure["affected"] + 0.5
) / (
    ct_structure["at_risk"] + 1
)

ct_structure["logit_event_rate"] = np.log(
    ct_structure["event_rate_smoothed"]
    /
    (
        1
        - ct_structure["event_rate_smoothed"]
    )
)


# =============================================================================
# FAERS PEDIATRIC VERSUS ADULT REPORTING ODDS RATIOS
# =============================================================================

faers_unique = (
    faers_rows[
        [
            "domain",
            "report_id",
            "std_smiles",
            "outcome",
        ]
    ]
    .drop_duplicates()
)

drug_totals = (
    faers_unique[
        faers_unique["outcome"]
        == "__ALL_REPORTS__"
    ][
        [
            "domain",
            "report_id",
            "std_smiles",
        ]
    ]
    .drop_duplicates()
    .groupby(
        [
            "domain",
            "std_smiles",
        ]
    )
    .size()
    .rename("all_reports")
    .reset_index()
)

event_counts = (
    faers_unique[
        faers_unique["outcome"]
        != "__ALL_REPORTS__"
    ]
    .groupby(
        [
            "domain",
            "std_smiles",
            "outcome",
        ]
    )
    .size()
    .rename("event_reports")
    .reset_index()
)

faers_counts = event_counts.merge(
    drug_totals,
    on=[
        "domain",
        "std_smiles",
    ],
    how="left",
)

faers_wide = faers_counts.pivot_table(
    index=[
        "std_smiles",
        "outcome",
    ],
    columns="domain",
    values=[
        "event_reports",
        "all_reports",
    ],
    aggfunc="first",
).reset_index()

faers_wide.columns = [
    "__".join(
        [
            str(part)
            for part in column
            if part
        ]
    )
    if isinstance(column, tuple)
    else column
    for column in faers_wide.columns
]

required_columns = [
    "event_reports__pediatric",
    "all_reports__pediatric",
    "event_reports__adult",
    "all_reports__adult",
]

for column in required_columns:
    if column not in faers_wide:
        faers_wide[column] = np.nan

faers_wide = faers_wide.dropna(
    subset=required_columns
).copy()

a = (
    faers_wide[
        "event_reports__pediatric"
    ]
    + 0.5
)

b = (
    faers_wide[
        "all_reports__pediatric"
    ]
    - faers_wide[
        "event_reports__pediatric"
    ]
    + 0.5
)

c = (
    faers_wide[
        "event_reports__adult"
    ]
    + 0.5
)

d = (
    faers_wide[
        "all_reports__adult"
    ]
    - faers_wide[
        "event_reports__adult"
    ]
    + 0.5
)

faers_wide[
    "pediatric_vs_adult_ror"
] = (
    (a / b)
    /
    (c / d)
)

faers_wide["log_ror"] = np.log(
    faers_wide[
        "pediatric_vs_adult_ror"
    ]
)

faers_wide["log_ror_se"] = np.sqrt(
    1 / a
    + 1 / b
    + 1 / c
    + 1 / d
)

faers_wide["log_ror_ci_low"] = (
    faers_wide["log_ror"]
    - 1.96
    * faers_wide["log_ror_se"]
)

faers_wide["log_ror_ci_high"] = (
    faers_wide["log_ror"]
    + 1.96
    * faers_wide["log_ror_se"]
)


# =============================================================================
# CANADA STRUCTURE-LEVEL OUTCOMES
# =============================================================================

if len(canada_rows):
    canada_structure = (
        canada_rows[
            [
                "report_id",
                "std_smiles",
                "outcome",
            ]
        ]
        .drop_duplicates()
        .groupby(
            [
                "std_smiles",
                "outcome",
            ]
        )
        .size()
        .rename("canada_event_reports")
        .reset_index()
    )

else:
    canada_structure = pd.DataFrame(
        columns=[
            "std_smiles",
            "outcome",
            "canada_event_reports",
        ]
    )


# =============================================================================
# SAVE SOURCE-SPECIFIC TABLES
# =============================================================================

atomic_csv(
    OUT /
    "clinicaltrials_structure_outcomes.csv",
    ct_structure,
)

atomic_csv(
    OUT /
    "faers_pediatric_vs_adult_ror.csv",
    faers_wide,
)

atomic_csv(
    OUT /
    "canada_structure_outcomes.csv",
    canada_structure,
)


# =============================================================================
# COMMON HARMONIZED LONG FORMAT
# =============================================================================

common_ct = ct_structure.assign(
    source="CLINICALTRIALS",
    estimate_type="smoothed_incidence",
    estimate=ct_structure[
        "event_rate_smoothed"
    ],
    denominator=ct_structure["at_risk"],
)[
    [
        "source",
        "std_smiles",
        "outcome",
        "estimate_type",
        "estimate",
        "denominator",
    ]
]

common_faers = faers_wide.assign(
    source="FAERS",
    estimate_type=(
        "pediatric_vs_adult_log_ROR"
    ),
    estimate=faers_wide["log_ror"],
    denominator=faers_wide[
        "all_reports__pediatric"
    ],
)[
    [
        "source",
        "std_smiles",
        "outcome",
        "estimate_type",
        "estimate",
        "denominator",
    ]
]

common_canada = canada_structure.assign(
    source="CANADA_VIGILANCE",
    estimate_type="pediatric_report_count",
    estimate=canada_structure[
        "canada_event_reports"
    ],
    denominator=np.nan,
)[
    [
        "source",
        "std_smiles",
        "outcome",
        "estimate_type",
        "estimate",
        "denominator",
    ]
]

harmonized = pd.concat(
    [
        common_ct,
        common_faers,
        common_canada,
    ],
    ignore_index=True,
)

atomic_csv(
    OUT /
    "harmonized_multisource_long.csv",
    harmonized,
)


# =============================================================================
# ENDPOINT FEASIBILITY CHECK
# =============================================================================

feasibility = (
    ct_structure
    .groupby("outcome")
    .agg(
        n_structures=(
            "std_smiles",
            "nunique",
        ),
        total_at_risk=(
            "at_risk",
            "sum",
        ),
        median_rate=(
            "event_rate_smoothed",
            "median",
        ),
    )
    .reset_index()
)

feasibility[
    "eligible_for_modeling"
] = (
    feasibility["n_structures"]
    >= MIN_ENDPOINT_STRUCTURES
)

atomic_csv(
    OUT /
    "endpoint_feasibility.csv",
    feasibility,
)

print("\nENDPOINT FEASIBILITY")
print("=" * 100)
print(feasibility.to_string(index=False))

eligible_endpoints = feasibility.loc[
    feasibility["eligible_for_modeling"],
    "outcome",
].tolist()

if not eligible_endpoints:
    raise RuntimeError(
        "No ClinicalTrials.gov endpoint has at "
        f"least {MIN_ENDPOINT_STRUCTURES} "
        "mapped structures. Review "
        "endpoint_feasibility.csv. Do not lower "
        "the threshold solely to obtain a result."
    )


# =============================================================================
# CHEMICAL REPRESENTATION
# =============================================================================

compound_table = ct_structure[
    ct_structure["outcome"].isin(
        eligible_endpoints
    )
].copy()

structures = sorted(
    compound_table[
        "std_smiles"
    ].unique()
)

molecules = [
    Chem.MolFromSmiles(smiles)
    for smiles in structures
]

fingerprint_generator = (
    rdFingerprintGenerator
    .GetMorganGenerator(
        radius=2,
        fpSize=2048,
    )
)

X_fp = np.asarray(
    [
        fingerprint_generator
        .GetFingerprintAsNumPy(molecule)
        for molecule in molecules
    ],
    dtype=np.float32,
)

descriptors = pd.DataFrame(
    [
        Descriptors.CalcMolDescriptors(
            molecule
        )
        for molecule in molecules
    ]
)

descriptors = descriptors.replace(
    [
        np.inf,
        -np.inf,
    ],
    np.nan,
)

for column in [
    "Ipc",
    "AvgIpc",
]:
    if column in descriptors.columns:
        descriptors[column] = np.log1p(
            descriptors[column].clip(
                lower=0
            )
        )

descriptors = descriptors.loc[
    :,
    descriptors.isna().mean() < 0.05,
]

descriptors = descriptors.fillna(
    descriptors.median()
)

descriptors = descriptors.loc[
    :,
    descriptors.std() > 0,
]

descriptors = descriptors.clip(
    -1e30,
    1e30,
)

X_chem = np.hstack(
    [
        X_fp,
        descriptors.to_numpy(
            dtype=np.float32
        ),
    ]
).astype(np.float32)

assert np.isfinite(X_chem).all()


# =============================================================================
# BIOLOGICAL SIGNATURES
# =============================================================================

bio_path = (
    CACHE /
    "clinicaltrials_X_bio.npy"
)

def signaturize_with_retry(space, smiles, attempts=8):
    """Retry BioFeat generation after temporary server/network failures."""

    for attempt in range(1, attempts + 1):
        try:
            print(f"BioFeat {space}: attempt {attempt}/{attempts}")

            block = (
                Signaturizer(space)
                .predict(smiles)
                .signature
                .astype(np.float32)
            )

            expected_shape = (len(smiles), 128)

            if block.shape != expected_shape:
                raise ValueError(
                    f"Unexpected shape for {space}: "
                    f"{block.shape}; expected {expected_shape}"
                )

            if not np.isfinite(block).all():
                raise ValueError(
                    f"BioFeat {space} contains NaN or infinity"
                )

            return block

        except Exception as error:
            if attempt == attempts:
                raise RuntimeError(
                    f"BioFeat {space} failed after {attempts} attempts. "
                    "Wait several minutes and rerun the cell. Previously "
                    "completed spaces remain saved in Google Drive."
                ) from error

            wait_seconds = min(
                60,
                5 * (2 ** (attempt - 1)),
            )

            print(
                f"Temporary BioFeat failure for {space}: "
                f"{type(error).__name__}: {error}"
            )

            print(
                f"Retrying in {wait_seconds} seconds. "
                "Completed spaces are already safe."
            )

            time.sleep(wait_seconds)

if bio_path.exists():
    X_bio = np.load(bio_path)

    print(
        "Loaded complete BioFeat matrix:",
        X_bio.shape,
    )

else:
    biological_blocks = []

    for space in SPACES:
        space_path = (
            CACHE /
            f"clinicaltrials_bio_{space}.npy"
        )

        if space_path.exists():
            block = np.load(space_path)

            print(
                "Loaded BioFeat space:",
                space,
            )

        else:
            block = (
                Signaturizer(space)
                .predict(structures)
                .signature
                .astype(np.float32)
            )

            atomic_npy(
                space_path,
                block,
            )

            print(
                "Saved BioFeat space:",
                space,
            )

        biological_blocks.append(block)

    X_bio = np.hstack(
        biological_blocks
    ).astype(np.float32)

    atomic_npy(
        bio_path,
        X_bio,
    )

assert X_bio.shape[0] == len(structures)
assert np.isfinite(X_bio).all()


# =============================================================================
# REPRESENTATIONS AND SCAFFOLDS
# =============================================================================

structure_index = {
    smiles: index
    for index, smiles in enumerate(
        structures
    )
}

scaffolds = []

for molecule_number, molecule in enumerate(
    molecules
):
    scaffold = (
        MurckoScaffold
        .MurckoScaffoldSmiles(
            mol=molecule,
            includeChirality=False,
        )
    )

    if not scaffold:
        scaffold = (
            f"NO_SCAFFOLD_{molecule_number}"
        )

    scaffolds.append(scaffold)

scaffolds = np.asarray(
    scaffolds,
    dtype=object,
)

representations = {
    "chem": X_chem,
    "bio_only": X_bio,
    "chem_true_bio": np.hstack(
        [
            X_chem,
            X_bio,
        ]
    ).astype(np.float32),
}

for permutation_seed in [
    101,
    202,
    303,
    404,
    505,
]:
    random_generator = (
        np.random.default_rng(
            permutation_seed
        )
    )

    permutation = (
        random_generator.permutation(
            len(X_bio)
        )
    )

    representations[
        f"chem_permuted_bio_{permutation_seed}"
    ] = np.hstack(
        [
            X_chem,
            X_bio[permutation],
        ]
    ).astype(np.float32)


# =============================================================================
# SCAFFOLD CROSS-VALIDATION
# =============================================================================

result_rows = []
prediction_rows = []

for endpoint in eligible_endpoints:
    endpoint_df = compound_table[
        compound_table["outcome"]
        == endpoint
    ].copy()

    matrix_indices = np.asarray(
        [
            structure_index[smiles]
            for smiles in endpoint_df[
                "std_smiles"
            ]
        ]
    )

    y_continuous = endpoint_df[
        "logit_event_rate"
    ].to_numpy(dtype=float)

    high_threshold = endpoint_df[
        "event_rate_smoothed"
    ].quantile(0.75)

    y_high = (
        endpoint_df[
            "event_rate_smoothed"
        ]
        >= high_threshold
    ).astype(int).to_numpy()

    unique_scaffolds = np.unique(
        scaffolds[matrix_indices]
    )

    if len(unique_scaffolds) < N_FOLDS:
        print(
            "Skipping endpoint because it has "
            "fewer than five unique scaffolds:",
            endpoint,
        )
        continue

    classification_allowed = (
        min(
            int(y_high.sum()),
            int(len(y_high) - y_high.sum()),
        )
        >= MIN_BINARY_CLASS
    )

    if not classification_allowed:
        print(
            "High-burden classification skipped "
            "for endpoint:",
            endpoint,
        )

    folds = list(
        GroupKFold(
            n_splits=N_FOLDS
        ).split(
            matrix_indices,
            groups=scaffolds[
                matrix_indices
            ],
        )
    )

    for (
        representation_name,
        feature_matrix,
    ) in representations.items():

        regression_predictions = (
            np.full(
                len(matrix_indices),
                np.nan,
                dtype=float,
            )
        )

        classification_predictions = (
            np.full(
                len(matrix_indices),
                np.nan,
                dtype=float,
            )
        )

        for fold_number, (
            train_local,
            test_local,
        ) in enumerate(folds):

            train_indices = matrix_indices[
                train_local
            ]

            test_indices = matrix_indices[
                test_local
            ]

            regression_path = (
                MODELS /
                (
                    f"{endpoint}__"
                    f"{representation_name}__"
                    f"fold{fold_number}__"
                    "reg.joblib"
                )
            )

            if regression_path.exists():
                regression_model = joblib.load(
                    regression_path
                )

                print(
                    "Loaded regression:",
                    endpoint,
                    representation_name,
                    fold_number + 1,
                )

            else:
                regression_model = XGBRegressor(
                    n_estimators=400,
                    max_depth=5,
                    learning_rate=0.04,
                    subsample=0.8,
                    colsample_bytree=0.6,
                    objective=(
                        "reg:squarederror"
                    ),
                    tree_method="hist",
                    n_jobs=-1,
                    random_state=(
                        SEED + fold_number
                    ),
                )

                regression_model.fit(
                    feature_matrix[
                        train_indices
                    ],
                    y_continuous[
                        train_local
                    ],
                )

                atomic_joblib(
                    regression_path,
                    regression_model,
                )

                print(
                    "Saved regression:",
                    endpoint,
                    representation_name,
                    fold_number + 1,
                )

            regression_predictions[
                test_local
            ] = regression_model.predict(
                feature_matrix[
                    test_indices
                ]
            )

            if (
                classification_allowed
                and len(
                    np.unique(
                        y_high[train_local]
                    )
                ) == 2
            ):
                classification_path = (
                    MODELS /
                    (
                        f"{endpoint}__"
                        f"{representation_name}__"
                        f"fold{fold_number}__"
                        "classification.joblib"
                    )
                )

                if classification_path.exists():
                    classification_model = (
                        joblib.load(
                            classification_path
                        )
                    )

                else:
                    positive_count = (
                        y_high[
                            train_local
                        ].sum()
                    )

                    scale_positive_weight = (
                        (
                            len(train_local)
                            - positive_count
                        )
                        / max(
                            positive_count,
                            1,
                        )
                    )

                    classification_model = (
                        XGBClassifier(
                            n_estimators=400,
                            max_depth=5,
                            learning_rate=0.04,
                            subsample=0.8,
                            colsample_bytree=0.6,
                            scale_pos_weight=(
                                scale_positive_weight
                            ),
                            eval_metric="logloss",
                            tree_method="hist",
                            n_jobs=-1,
                            random_state=(
                                SEED
                                + fold_number
                            ),
                        )
                    )

                    classification_model.fit(
                        feature_matrix[
                            train_indices
                        ],
                        y_high[
                            train_local
                        ],
                    )

                    atomic_joblib(
                        classification_path,
                        classification_model,
                    )

                classification_predictions[
                    test_local
                ] = (
                    classification_model
                    .predict_proba(
                        feature_matrix[
                            test_indices
                        ]
                    )[:, 1]
                )

        spearman_value = spearmanr(
            y_continuous,
            regression_predictions,
        ).statistic

        result = {
            "endpoint": endpoint,
            "representation": (
                representation_name
            ),
            "n": len(y_continuous),
            "spearman": spearman_value,
            "rmse": (
                mean_squared_error(
                    y_continuous,
                    regression_predictions,
                )
                ** 0.5
            ),
            "mae": mean_absolute_error(
                y_continuous,
                regression_predictions,
            ),
            "auroc_high_burden": np.nan,
            "auprc_high_burden": np.nan,
        }

        if (
            np.isfinite(
                classification_predictions
            ).all()
            and len(
                np.unique(y_high)
            ) == 2
        ):
            result[
                "auroc_high_burden"
            ] = roc_auc_score(
                y_high,
                classification_predictions,
            )

            result[
                "auprc_high_burden"
            ] = average_precision_score(
                y_high,
                classification_predictions,
            )

        result_rows.append(result)

        for row_number, smiles in enumerate(
            endpoint_df["std_smiles"]
        ):
            prediction_rows.append(
                {
                    "endpoint": endpoint,
                    "std_smiles": smiles,
                    "representation": (
                        representation_name
                    ),
                    "observed_logit_rate": (
                        y_continuous[
                            row_number
                        ]
                    ),
                    "regression_oof": (
                        regression_predictions[
                            row_number
                        ]
                    ),
                    "high_burden": (
                        y_high[row_number]
                    ),
                    "classification_oof": (
                        classification_predictions[
                            row_number
                        ]
                    ),
                }
            )

        atomic_csv(
            OUT /
            "representation_results.csv",
            pd.DataFrame(result_rows),
        )

        atomic_csv(
            OUT /
            "oof_predictions.csv",
            pd.DataFrame(
                prediction_rows
            ),
        )

        print(
            endpoint,
            representation_name,
            result,
        )


results = pd.DataFrame(result_rows)
predictions = pd.DataFrame(
    prediction_rows
)


# =============================================================================
# PAIRED COMPOUND BOOTSTRAP
# =============================================================================

bootstrap_rows = []

for endpoint in eligible_endpoints:
    chemical_predictions = predictions[
        (
            predictions["endpoint"]
            == endpoint
        )
        & (
            predictions["representation"]
            == "chem"
        )
    ]

    multimodal_predictions = predictions[
        (
            predictions["endpoint"]
            == endpoint
        )
        & (
            predictions["representation"]
            == "chem_true_bio"
        )
    ]

    paired = chemical_predictions.merge(
        multimodal_predictions,
        on=[
            "endpoint",
            "std_smiles",
            "observed_logit_rate",
        ],
        suffixes=(
            "_chem",
            "_bio",
        ),
    )

    if len(paired) < MIN_ENDPOINT_STRUCTURES:
        continue

    random_generator = (
        np.random.default_rng(SEED)
    )

    differences = []
    sample_size = len(paired)

    for bootstrap_number in range(N_BOOT):
        sampled_indices = (
            random_generator.integers(
                0,
                sample_size,
                sample_size,
            )
        )

        sampled = paired.iloc[
            sampled_indices
        ]

        chemical_spearman = spearmanr(
            sampled["observed_logit_rate"],
            sampled[
                "regression_oof_chem"
            ],
        ).statistic

        bio_spearman = spearmanr(
            sampled["observed_logit_rate"],
            sampled[
                "regression_oof_bio"
            ],
        ).statistic

        if (
            np.isfinite(
                chemical_spearman
            )
            and np.isfinite(
                bio_spearman
            )
        ):
            differences.append(
                bio_spearman
                - chemical_spearman
            )

    differences = np.asarray(
        differences
    )

    chemical_point = spearmanr(
        paired["observed_logit_rate"],
        paired["regression_oof_chem"],
    ).statistic

    bio_point = spearmanr(
        paired["observed_logit_rate"],
        paired["regression_oof_bio"],
    ).statistic

    point_difference = (
        bio_point - chemical_point
    )

    p_lower = (
        np.sum(differences <= 0) + 1
    ) / (
        len(differences) + 1
    )

    p_upper = (
        np.sum(differences >= 0) + 1
    ) / (
        len(differences) + 1
    )

    bootstrap_rows.append(
        {
            "endpoint": endpoint,
            "metric": "spearman",
            "true_bio_minus_chem": (
                point_difference
            ),
            "ci_low": np.percentile(
                differences,
                2.5,
            ),
            "ci_high": np.percentile(
                differences,
                97.5,
            ),
            "p_two_sided": min(
                1,
                2 * min(
                    p_lower,
                    p_upper,
                ),
            ),
            "n_boot": len(
                differences
            ),
        }
    )

    atomic_csv(
        OUT /
        "paired_bootstrap.csv",
        pd.DataFrame(
            bootstrap_rows
        ),
    )


# =============================================================================
# CROSS-SOURCE COVERAGE
# =============================================================================

coverage = (
    harmonized
    .groupby(
        [
            "source",
            "outcome",
        ]
    )
    .agg(
        n_structures=(
            "std_smiles",
            "nunique",
        ),
        n_rows=(
            "std_smiles",
            "size",
        ),
    )
    .reset_index()
)

atomic_csv(
    OUT /
    "cross_source_coverage.csv",
    coverage,
)


# =============================================================================
# FINAL OUTPUT
# =============================================================================

print("\nANALYSIS COMPLETE")
print("=" * 100)

print("\nResults folder:")
print(OUT)

print("\nPrimary representation results:")
print(
    results.to_string(
        index=False
    )
)

print("\nCross-source coverage:")
print(
    coverage.to_string(
        index=False
    )
)

print("\nMain saved files:")

for output_file in [
    "endpoint_feasibility.csv",
    "clinicaltrials_structure_outcomes.csv",
    "faers_pediatric_vs_adult_ror.csv",
    "canada_structure_outcomes.csv",
    "harmonized_multisource_long.csv",
    "representation_results.csv",
    "oof_predictions.csv",
    "paired_bootstrap.csv",
    "cross_source_coverage.csv",
]:
    print(" ", OUT / output_file)

print(
    "\nIMPORTANT: ClinicalTrials.gov incidence, "
    "FAERS reporting odds ratios, and Canada "
    "report counts were harmonized to the same "
    "chemical structures and outcome categories, "
    "but they were not pooled as if they measured "
    "the same quantity."
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved pipeline code to: /content/drive/MyDrive/pediatric_clinical_safety_pipeline/pediatric_clinical_multisource_pipeline.py
Resuming ClinicalTrials.gov at 10,000 studies
ClinicalTrials.gov studies ready: 10000
Loaded harmonized trial rows: 8099
Clinical-trial event rows: 8099
Clinical-trial unique drug names: 882


/tmp/ipykernel_785/3193127235.py:1070: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  faers_rows = pd.read_csv(


Loaded harmonized FAERS rows: 135066
FAERS harmonized rows: 135066
FAERS unique drug names: 8405
Canada validation skipped. The optional file was not found: /content/drive/MyDrive/pediatric_clinical_safety_pipeline/raw/canada_vigilance.csv
PubChem mapping 25/9282
PubChem mapping 50/9282
PubChem mapping 75/9282
PubChem mapping 100/9282
PubChem mapping 125/9282
PubChem mapping 150/9282
PubChem mapping 175/9282
PubChem mapping 200/9282
PubChem mapping 225/9282
PubChem mapping 250/9282
PubChem mapping 275/9282
PubChem mapping 300/9282
PubChem mapping 325/9282
PubChem mapping 350/9282
PubChem mapping 375/9282
PubChem mapping 400/9282
PubChem mapping 425/9282
PubChem mapping 450/9282
PubChem mapping 475/9282
PubChem mapping 500/9282
PubChem mapping 525/9282
PubChem mapping 550/9282
PubChem mapping 575/9282
PubChem mapping 600/9282
PubChem mapping 625/9282
PubChem mapping 650/9282
PubChem mapping 675/9282
PubChem mapping 700/9282
PubChem mapping 725/9282
PubChem mapping 750/9282
PubChem mappi

RemoteDisconnected: Remote end closed connection without response

In [ ]:
# =============================================================================
# RESCUE CELL — FINISH BIOFEAT WITH RETRIES AND DRIVE CHECKPOINTS
# =============================================================================

import time
import numpy as np
from pathlib import Path
from signaturizer import Signaturizer

# These variables should already exist from the interrupted main cell.
required_variables = [
    "CACHE",
    "structures",
    "SPACES",
    "atomic_npy",
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise RuntimeError(
        "The previous runtime state was lost. Rerun the corrected complete "
        f"pipeline instead. Missing variables: {missing_variables}"
    )


def signaturize_with_retry(
    space,
    smiles,
    attempts=10,
):
    for attempt in range(1, attempts + 1):
        try:
            print(
                f"\nBioFeat {space}: "
                f"attempt {attempt}/{attempts}"
            )

            signaturizer = Signaturizer(space)

            block = (
                signaturizer
                .predict(smiles)
                .signature
                .astype(np.float32)
            )

            expected_shape = (
                len(smiles),
                128,
            )

            if block.shape != expected_shape:
                raise ValueError(
                    f"Incorrect shape for {space}: "
                    f"{block.shape}; expected {expected_shape}"
                )

            if not np.isfinite(block).all():
                raise ValueError(
                    f"{space} contains NaN or infinity"
                )

            print(
                f"BioFeat {space} completed:",
                block.shape,
            )

            return block

        except Exception as error:
            print(
                f"BioFeat {space} failed with "
                f"{type(error).__name__}: {error}"
            )

            if attempt == attempts:
                raise RuntimeError(
                    f"BioFeat {space} failed after "
                    f"{attempts} attempts. Wait 10–20 minutes "
                    "and rerun this rescue cell. Previously completed "
                    "spaces remain saved in Drive."
                ) from error

            wait_seconds = min(
                60,
                5 * (2 ** (attempt - 1)),
            )

            print(
                f"Waiting {wait_seconds} seconds "
                "before retrying..."
            )

            time.sleep(wait_seconds)


bio_blocks = []

for space in SPACES:
    space_path = (
        Path(CACHE)
        / f"clinicaltrials_bio_{space}.npy"
    )

    load_existing = False

    if space_path.exists():
        try:
            block = np.load(space_path)

            if (
                block.shape == (len(structures), 128)
                and np.isfinite(block).all()
            ):
                load_existing = True

                print(
                    f"BioFeat {space}: loaded saved checkpoint",
                    block.shape,
                )

            else:
                print(
                    f"BioFeat {space}: saved file failed validation; "
                    "regenerating it"
                )

        except Exception as error:
            print(
                f"BioFeat {space}: checkpoint could not be loaded; "
                f"regenerating it ({error})"
            )

    if not load_existing:
        block = signaturize_with_retry(
            space,
            structures,
            attempts=10,
        )

        atomic_npy(
            space_path,
            block,
        )

        print(
            f"BioFeat {space}: saved to Google Drive"
        )

    bio_blocks.append(block)


X_bio = np.hstack(
    bio_blocks
).astype(np.float32)

assert X_bio.shape == (
    len(structures),
    len(SPACES) * 128,
)

assert np.isfinite(X_bio).all()

complete_bio_path = (
    Path(CACHE)
    / "clinicaltrials_X_bio.npy"
)

atomic_npy(
    complete_bio_path,
    X_bio,
)

print("\n" + "=" * 100)
print("ALL BIOFEAT SPACES COMPLETE")
print("X_bio shape:", X_bio.shape)
print("Saved complete matrix to:")
print(complete_bio_path)
print("=" * 100)


BioFeat B1: attempt 1/10
BioFeat B1 failed with RemoteDisconnected: Remote end closed connection without response
Waiting 5 seconds before retrying...

BioFeat B1: attempt 2/10
BioFeat B1 failed with RemoteDisconnected: Remote end closed connection without response
Waiting 10 seconds before retrying...

BioFeat B1: attempt 3/10
BioFeat B1 failed with RemoteDisconnected: Remote end closed connection without response
Waiting 20 seconds before retrying...

BioFeat B1: attempt 4/10
BioFeat B1 failed with RemoteDisconnected: Remote end closed connection without response
Waiting 40 seconds before retrying...

BioFeat B1: attempt 5/10
BioFeat B1 failed with RemoteDisconnected: Remote end closed connection without response
Waiting 60 seconds before retrying...

BioFeat B1: attempt 6/10
BioFeat B1 failed with RemoteDisconnected: Remote end closed connection without response
Waiting 60 seconds before retrying...

BioFeat B1: attempt 7/10
BioFeat B1 failed with RemoteDisconnected: Remote end clo

RuntimeError: BioFeat B1 failed after 10 attempts. Wait 10–20 minutes and rerun this rescue cell. Previously completed spaces remain saved in Drive.